<a href="https://colab.research.google.com/github/Archisman936/amazon_ml_2026/blob/main/blocking_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# AMAZON ML CHALLENGE 2026
# STAGE 4 — BLOCKING / CANDIDATE GENERATION
# ============================================================
#
# FINAL FIXED VERSION
#
# FIXES:
# 1. Exact user-provided Stage-3 paths
# 2. quote='' to disable CSV quote parsing
# 3. Explicit VARCHAR schema from TSV header
# 4. NO new_line parameter
# 5. DuckDB database on Colab local storage
# 6. DuckDB temp/spill storage on Colab local storage
# 7. Final outputs saved to Google Drive
# 8. No expected row-count validation
# 9. Correct parameterized SQL
#
# ============================================================


# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================

!pip -q install duckdb scikit-learn pandas numpy pyarrow psutil


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import time
import shutil
import warnings

import duckdb
import numpy as np
import pandas as pd
import psutil

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")


# ============================================================
# 2. DIRECTORIES
# ============================================================

TRAIN_STAGE3_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/train_normalized"
)

TEST_STAGE3_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/test_normalized"
)


# ============================================================
# 3. GROUND TRUTH
# ============================================================

GROUND_TRUTH_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/train_normalized/"
    "Copy of Copy of train_ground_truth.tsv"
)


# ============================================================
# 4. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/dataset"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 5. LOCAL DUCKDB WORKSPACE
# ============================================================
#
# IMPORTANT:
# DuckDB database and temporary spill files stay on local
# Colab storage.
#
# This avoids Google Drive DuckDB temp-file errors.
# ============================================================

LOCAL_WORK_DIR = (
    "/content/amazon_ml_stage4_work"
)

LOCAL_TEMP_DIR = os.path.join(
    LOCAL_WORK_DIR,
    "duckdb_temp"
)

DB_PATH = os.path.join(
    LOCAL_WORK_DIR,
    "stage4_blocking.duckdb"
)

os.makedirs(
    LOCAL_WORK_DIR,
    exist_ok=True
)

os.makedirs(
    LOCAL_TEMP_DIR,
    exist_ok=True
)


# ============================================================
# 6. REMOVE OLD LOCAL DATABASE
# ============================================================
#
# Only the LOCAL Stage-4 database is removed.
# Google Drive inputs and outputs are untouched.
# ============================================================

if os.path.exists(DB_PATH):

    try:

        os.remove(
            DB_PATH
        )

    except Exception:

        pass


# ============================================================
# 7. BLOCKING SETTINGS
# ============================================================

MAX_BLOCK_SIZE = 1000


# ============================================================
# 8. TF-IDF SETTINGS
# ============================================================

TFIDF_TOP_K = 15

TFIDF_MAX_FEATURES = 50_000

TFIDF_BUCKET_MAX_ROWS = 120_000

TFIDF_MIN_DF = 1


# ============================================================
# 9. TF-IDF THRESHOLD
# ============================================================

MAX_DETERMINISTIC_CANDIDATES = 30


# ============================================================
# 10. CPU / MEMORY SETTINGS
# ============================================================

CPU_COUNT = (
    os.cpu_count()
    or
    2
)

DUCKDB_THREADS = max(
    2,
    min(
        8,
        CPU_COUNT
    )
)

RAM_GB = (
    psutil.virtual_memory().total
    /
    (1024 ** 3)
)

DUCKDB_MEMORY_GB = max(
    4,
    int(
        RAM_GB * 0.70
    )
)


print(
    "CPU cores:",
    CPU_COUNT
)

print(
    "DuckDB threads:",
    DUCKDB_THREADS
)

print(
    "RAM:",
    f"{RAM_GB:.1f} GB"
)

print(
    "DuckDB memory limit:",
    f"{DUCKDB_MEMORY_GB} GB"
)


# ============================================================
# 11. EXACT INPUT FILE PATHS
# ============================================================

S1_TRAIN = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "train_normalized/Copy of train_source1_stage3_normalized.tsv"
)

S2_TRAIN = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "train_normalized/Copy of train_source2_stage3_normalized.tsv"
)

S3_TRAIN = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "train_normalized/Copy of train_source3_stage3_normalized.tsv"
)


S1_TEST = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of test_source1_stage3_normalized.tsv"
)

S2_TEST = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of test_source2_stage3_normalized.tsv"
)

S3_TEST = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of Copy of test_source3_stage3_normalized.tsv"
)


GT_TRAIN = (
    GROUND_TRUTH_PATH
)


# ============================================================
# 12. PRINT PATHS
# ============================================================

print(
    "\n"
    "========== INPUT PATHS ==========\n"
)

print(
    "S1 TRAIN:"
)

print(
    S1_TRAIN
)

print(
    "\nS2 TRAIN:"
)

print(
    S2_TRAIN
)

print(
    "\nS3 TRAIN:"
)

print(
    S3_TRAIN
)

print(
    "\nS1 TEST:"
)

print(
    S1_TEST
)

print(
    "\nS2 TEST:"
)

print(
    S2_TEST
)

print(
    "\nS3 TEST:"
)

print(
    S3_TEST
)

print(
    "\nGROUND TRUTH:"
)

print(
    GT_TRAIN
)

print(
    "\nOUTPUT DIRECTORY:"
)

print(
    OUTPUT_DIR
)

print(
    "\nLOCAL DUCKDB:"
)

print(
    DB_PATH
)

print(
    "\nLOCAL DUCKDB TEMP:"
)

print(
    LOCAL_TEMP_DIR
)


# ============================================================
# 13. FILE EXISTENCE CHECK
# ============================================================
#
# NO ROW-COUNT COMPLETENESS CHECK.
# ============================================================

required_files = [

    S1_TRAIN,

    S2_TRAIN,

    S3_TRAIN,

    S1_TEST,

    S2_TEST,

    S3_TEST,

    GT_TRAIN

]


missing_files = [

    path

    for path in required_files

    if not os.path.exists(path)

]


if missing_files:

    print(
        "\nMissing files:\n"
    )

    for path in missing_files:

        print(
            " -",
            path
        )

    raise FileNotFoundError(
        "\nRequired input files are missing."
    )


print(
    "\nAll required files found."
)

print(
    "Stage-3 expected-row-count check: DISABLED"
)


# ============================================================
# 14. HELPER FUNCTIONS
# ============================================================

def escape_sql_string(
    value
):

    return str(
        value
    ).replace(
        "'",
        "''"
    )


def sql_ident(
    name
):

    return (
        '"'
        +
        str(name).replace(
            '"',
            '""'
        )
        +
        '"'
    )


def sql_literal(
    value
):

    return (
        "'"
        +
        str(value).replace(
            "'",
            "''"
        )
        +
        "'"
    )


def execute_timed(
    con,
    sql,
    label,
    parameters=None
):

    print(
        f"\n[START] {label}"
    )

    start = time.time()


    if parameters is None:

        result = con.execute(
            sql
        )

    else:

        result = con.execute(
            sql,
            parameters
        )


    elapsed = (
        time.time()
        -
        start
    )


    print(
        f"[DONE] {label} "
        f"({elapsed / 60:.1f} min)"
    )


    return result


def get_columns(
    con,
    table_name
):

    rows = con.execute(

        f"""
        DESCRIBE
            {sql_ident(table_name)}
        """

    ).fetchall()


    return [

        row[0]

        for row in rows

    ]


def find_column(
    columns,
    candidates,
    required=False
):

    lookup = {

        column.lower():
        column

        for column in columns

    }


    for candidate in candidates:

        if (
            candidate.lower()
            in
            lookup
        ):

            return lookup[
                candidate.lower()
            ]


    if required:

        raise KeyError(

            "None of these columns "
            "were found: "
            +
            str(candidates)

        )


    return None


def clean_sql_value(
    column_name
):

    if column_name is None:

        return "''"


    return (

        f"lower("
        f"trim("
        f"coalesce("
        f"{sql_ident(column_name)},"
        f"''"
        f")"
        f")"
        f")"

    )


# ============================================================
# 15. READ TSV HEADER
# ============================================================

def read_tsv_header(
    path
):

    with open(

        path,

        "r",

        encoding="utf-8-sig",

        newline=""

    ) as file:

        first_line = file.readline()


    if not first_line:

        raise ValueError(
            f"Empty file: {path}"
        )


    header = (

        first_line

        .rstrip(
            "\r\n"
        )

        .split(
            "\t"
        )

    )


    header = [

        column.strip()

        for column in header

    ]


    if len(header) < 2:

        raise ValueError(

            f"Malformed TSV header: "
            f"{path}\n"
            f"Detected header: "
            f"{header}"

        )


    duplicate_headers = [

        column

        for column in set(
            header
        )

        if header.count(
            column
        ) > 1

    ]


    if duplicate_headers:

        raise ValueError(

            "Duplicate TSV headers "
            f"in {path}: "
            +
            str(
                duplicate_headers
            )

        )


    return header


# ============================================================
# 16. EXPLICIT VARCHAR SCHEMA
# ============================================================

def build_columns_sql(
    header
):

    parts = []


    for column in header:

        parts.append(

            f"{sql_literal(column)}: "
            f"'VARCHAR'"

        )


    return (

        "{"

        +

        ", ".join(parts)

        +

        "}"

    )


# ============================================================
# 17. ROBUST TSV LOADER
# ============================================================
#
# MAIN FIX:
#
# quote=''
#
# This disables CSV-style quote parsing.
#
# IMPORTANT:
# There is intentionally NO `new_line=` parameter.
#
# DuckDB handles newline detection itself.
# ============================================================

def load_tsv(

    con,

    table_name,

    path

):

    print(

        f"\n========== "
        f"LOADING {table_name} "
        f"=========="

    )


    print(
        path
    )


    # --------------------------------------------------------
    # Read header
    # --------------------------------------------------------

    header = read_tsv_header(
        path
    )


    print(

        "Detected columns:",

        len(header)

    )


    print(

        "First columns:",

        header[
            :min(
                8,
                len(header)
            )
        ]

    )


    # --------------------------------------------------------
    # Explicit schema
    # --------------------------------------------------------

    columns_sql = (
        build_columns_sql(
            header
        )
    )


    # --------------------------------------------------------
    # DuckDB TSV reader
    #
    # NOTE:
    # new_line is deliberately omitted.
    # --------------------------------------------------------

    sql = f"""

    CREATE OR REPLACE TABLE
        {sql_ident(table_name)}
    AS

    SELECT *

    FROM read_csv(

        '{escape_sql_string(path)}',

        auto_detect=false,

        delim='\\\\t',

        header=true,

        columns={columns_sql},

        quote='',

        all_varchar=true,

        null_padding=true,

        strict_mode=true,

        ignore_errors=false,

        max_line_size=10000000

    );

    """


    execute_timed(

        con,

        sql,

        f"Load {table_name}"

    )


    count = con.execute(

        f"""

        SELECT COUNT(*)

        FROM
            {sql_ident(table_name)}

        """

    ).fetchone()[0]


    print(

        f"{table_name}: "
        f"{count:,} rows"

    )


    return count


# ============================================================
# 18. CONNECT TO DUCKDB
# ============================================================

print(

    "\n"
    "========== "
    "CONNECTING TO LOCAL DUCKDB "
    "=========="

)


con = duckdb.connect(
    DB_PATH
)


con.execute(
    f"PRAGMA threads={DUCKDB_THREADS}"
)


con.execute(

    f"""
    PRAGMA memory_limit=
    '{DUCKDB_MEMORY_GB}GB'
    """

)


con.execute(

    f"""
    SET temp_directory=
    '{escape_sql_string(LOCAL_TEMP_DIR)}'
    """

)


con.execute(
    "SET preserve_insertion_order=false"
)


con.execute(
    "PRAGMA enable_progress_bar=false"
)


print(

    "DuckDB database:",

    DB_PATH

)


print(

    "DuckDB temp:",

    LOCAL_TEMP_DIR

)


# ============================================================
# 19. LOAD TRAIN DATA
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "LOADING TRAIN DATA"

)

print(

    "============================================================"

)


load_tsv(

    con,

    "train_s1",

    S1_TRAIN

)


load_tsv(

    con,

    "train_s2",

    S2_TRAIN

)


load_tsv(

    con,

    "train_s3",

    S3_TRAIN

)


load_tsv(

    con,

    "train_gt",

    GT_TRAIN

)


# ============================================================
# 20. LOAD TEST DATA
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "LOADING TEST DATA"

)

print(

    "============================================================"

)


load_tsv(

    con,

    "test_s1",

    S1_TEST

)


load_tsv(

    con,

    "test_s2",

    S2_TEST

)


load_tsv(

    con,

    "test_s3",

    S3_TEST

)


# ============================================================
# 21. DETECT COLUMNS
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "DETECTING STAGE-3 COLUMNS"

)

print(

    "============================================================"

)


table_columns = {}


for table_name in [

    "train_s1",

    "train_s2",

    "train_s3",

    "test_s1",

    "test_s2",

    "test_s3"

]:

    table_columns[
        table_name
    ] = get_columns(

        con,

        table_name

    )


# ============================================================
# 22. COLUMN DETECTION
# ============================================================

def find_id_column(
    columns
):

    return find_column(

        columns,

        [

            "entity_id",

            "source1_entity_id",

            "source2_entity_id",

            "source3_entity_id"

        ],

        required=True

    )


def find_name_column(
    columns
):

    return find_column(

        columns,

        [

            # ACTUAL STAGE-3 FIELD
            "business_name_normalized_original",

            # fallback
            "business_name_normalized",

            "business_name_norm",

            "business_name_clean_normalized"

        ],

        required=True

    )


def find_roman_column(
    columns
):

    return find_column(

        columns,

        [

            "business_name_normalized_roman",

            "business_name_roman_search",

            "business_name_roman"

        ],

        required=True

    )


def find_phonetic_column(
    columns
):

    return find_column(

        columns,

        [

            "business_name_roman_phonetic",

            "business_name_phonetic"

        ],

        required=False

    )


def find_country_column(
    columns
):

    return find_column(

        columns,

        [

            "country",

            "country_normalized"

        ],

        required=True

    )


def detect_address_columns(
    columns
):

    return {

        "postal":

            find_column(

                columns,

                [

                    "address_postal_code",

                    "business_address_postal_code",

                    "postal_code"

                ],

                required=False

            ),


        "house":

            find_column(

                columns,

                [

                    "address_house_number",

                    "business_address_house_number",

                    "house_number"

                ],

                required=False

            ),


        "locality":

            find_column(

                columns,

                [

                    "address_locality",

                    "business_address_locality",

                    "locality"

                ],

                required=False

            ),


        "city":

            find_column(

                columns,

                [

                    "address_city",

                    "business_address_city",

                    "city"

                ],

                required=False

            )

    }


# ============================================================
# 23. BUILD COLUMN MAP
# ============================================================

column_map = {}


for table_name in [

    "train_s1",

    "train_s2",

    "train_s3",

    "test_s1",

    "test_s2",

    "test_s3"

]:

    columns = table_columns[
        table_name
    ]


    column_map[
        table_name
    ] = {

        "id":

            find_id_column(
                columns
            ),


        "name":

            find_name_column(
                columns
            ),


        "roman":

            find_roman_column(
                columns
            ),


        "phonetic":

            find_phonetic_column(
                columns
            ),


        "country":

            find_country_column(
                columns
            ),


        "address":

            detect_address_columns(
                columns
            )

    }


# ============================================================
# 24. PRINT COLUMN MAP
# ============================================================

for table_name in [

    "train_s1",

    "train_s2",

    "train_s3",

    "test_s1",

    "test_s2",

    "test_s3"

]:

    mapping = column_map[
        table_name
    ]


    print(
        f"\n{table_name}"
    )


    print(
        "ID       :",
        mapping["id"]
    )


    print(
        "NAME     :",
        mapping["name"]
    )


    print(
        "ROMAN    :",
        mapping["roman"]
    )


    print(
        "PHONETIC :",
        mapping["phonetic"]
    )


    print(
        "COUNTRY  :",
        mapping["country"]
    )


    print(
        "ADDRESS  :",
        mapping["address"]
    )


# ============================================================
# 25. CREATE FEATURE TABLE
# ============================================================

def create_feature_table(

    con,

    source_table,

    output_table,

    mapping

):

    id_col = mapping[
        "id"
    ]

    name_col = mapping[
        "name"
    ]

    roman_col = mapping[
        "roman"
    ]

    phonetic_col = mapping[
        "phonetic"
    ]

    country_col = mapping[
        "country"
    ]

    address_info = mapping[
        "address"
    ]


    name_expr = clean_sql_value(
        name_col
    )

    roman_expr = clean_sql_value(
        roman_col
    )

    phonetic_expr = clean_sql_value(
        phonetic_col
    )

    country_expr = clean_sql_value(
        country_col
    )

    postal_expr = clean_sql_value(
        address_info["postal"]
    )

    house_expr = clean_sql_value(
        address_info["house"]
    )

    locality_expr = clean_sql_value(
        address_info["locality"]
    )

    city_expr = clean_sql_value(
        address_info["city"]
    )


    sql = f"""

    CREATE OR REPLACE TABLE
        {sql_ident(output_table)}
    AS

    SELECT

        CAST(
            {sql_ident(id_col)}
            AS VARCHAR
        )
        AS entity_id,


        {name_expr}
        AS name_key_base,


        {roman_expr}
        AS roman_name_key_base,


        {phonetic_expr}
        AS phonetic_key_base,


        {country_expr}
        AS country_key,


        {postal_expr}
        AS postal_key,


        {house_expr}
        AS house_key,


        {locality_expr}
        AS locality_key,


        {city_expr}
        AS city_key,


        concat_ws(

            '¦',

            nullif(
                {house_expr},
                ''
            ),

            nullif(
                {locality_expr},
                ''
            )

        )
        AS house_locality_key,


        concat_ws(

            '¦',

            nullif(
                {house_expr},
                ''
            ),

            nullif(
                {city_expr},
                ''
            )

        )
        AS house_city_key,


        concat_ws(

            '¦',

            nullif(
                {name_expr},
                ''
            ),

            nullif(
                {country_expr},
                ''
            )

        )
        AS name_country_key,


        concat_ws(

            '¦',

            nullif(
                {name_expr},
                ''
            ),

            nullif(
                {postal_expr},
                ''
            )

        )
        AS name_postal_key,


        regexp_extract(

            {roman_expr},

            '([a-z0-9]{{2}})',

            1

        )
        AS tfidf_prefix2


    FROM
        {sql_ident(source_table)};

    """


    execute_timed(

        con,

        sql,

        f"Create {output_table}"

    )


# ============================================================
# 26. TRAIN FEATURE TABLES
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "CREATING TRAIN FEATURE TABLES"

)

print(

    "============================================================"

)


create_feature_table(

    con,

    "train_s1",

    "f_train_s1",

    column_map["train_s1"]

)


create_feature_table(

    con,

    "train_s2",

    "f_train_s2",

    column_map["train_s2"]

)


create_feature_table(

    con,

    "train_s3",

    "f_train_s3",

    column_map["train_s3"]

)


# ============================================================
# 27. TEST FEATURE TABLES
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "CREATING TEST FEATURE TABLES"

)

print(

    "============================================================"

)


create_feature_table(

    con,

    "test_s1",

    "f_test_s1",

    column_map["test_s1"]

)


create_feature_table(

    con,

    "test_s2",

    "f_test_s2",

    column_map["test_s2"]

)


create_feature_table(

    con,

    "test_s3",

    "f_test_s3",

    column_map["test_s3"]

)


# ============================================================
# 28. SHOW FEATURE TABLE COUNTS
# ============================================================

print(

    "\n"
    "========== FEATURE TABLE COUNTS =========="

)


for table_name in [

    "f_train_s1",

    "f_train_s2",

    "f_train_s3",

    "f_test_s1",

    "f_test_s2",

    "f_test_s3"

]:

    count = con.execute(

        f"""

        SELECT COUNT(*)

        FROM
            {table_name}

        """

    ).fetchone()[0]


    print(

        f"{table_name}: "
        f"{count:,} rows"

    )


# ============================================================
# 29. CREATE CANDIDATE TABLES
# ============================================================

con.execute(

    """
    CREATE OR REPLACE TABLE
        train_candidate_long
    (

        source1_entity_id VARCHAR,

        candidate_entity_id VARCHAR,

        source_pair VARCHAR,

        strategy VARCHAR

    )
    """

)


con.execute(

    """
    CREATE OR REPLACE TABLE
        test_candidate_long
    (

        source1_entity_id VARCHAR,

        candidate_entity_id VARCHAR,

        source_pair VARCHAR,

        strategy VARCHAR

    )
    """

)


# ============================================================
# 30. EXACT BLOCK
# ============================================================

def add_exact_block(

    con,

    s1_table,

    sx_table,

    target_table,

    source_pair,

    key_column,

    strategy

):

    print(

        f"\nRunning block: "
        f"{strategy} "
        f"[{source_pair}]"

    )


    sql = f"""

    INSERT INTO
        {target_table}


    SELECT

        a.entity_id
        AS source1_entity_id,


        b.entity_id
        AS candidate_entity_id,


        '{escape_sql_string(source_pair)}'
        AS source_pair,


        '{escape_sql_string(strategy)}'
        AS strategy


    FROM

        {s1_table} a


    INNER JOIN

        {sx_table} b


        ON


        a.{sql_ident(key_column)}
        <>
        ''


        AND


        a.{sql_ident(key_column)}
        =
        b.{sql_ident(key_column)}


    WHERE


        a.{sql_ident(key_column)}


        IN


        (


            SELECT

                {sql_ident(key_column)}


            FROM

                {s1_table}


            WHERE

                {sql_ident(key_column)}
                <>
                ''


            GROUP BY

                {sql_ident(key_column)}


            HAVING

                COUNT(*)
                <=
                {MAX_BLOCK_SIZE}


        )


        AND


        b.{sql_ident(key_column)}


        IN


        (


            SELECT

                {sql_ident(key_column)}


            FROM

                {sx_table}


            WHERE

                {sql_ident(key_column)}
                <>
                ''


            GROUP BY

                {sql_ident(key_column)}


            HAVING

                COUNT(*)
                <=
                {MAX_BLOCK_SIZE}


        );

    """


    execute_timed(

        con,

        sql,

        f"{strategy} "
        f"[{source_pair}]"

    )


# ============================================================
# 31. HOUSE + LOCATION BLOCK
# ============================================================

def add_house_location_block(

    con,

    s1_table,

    sx_table,

    target_table,

    source_pair

):

    print(

        f"\nRunning block: "
        f"house_locality "
        f"[{source_pair}]"

    )


    sql = f"""

    INSERT INTO
        {target_table}


    SELECT

        a.entity_id
        AS source1_entity_id,


        b.entity_id
        AS candidate_entity_id,


        '{escape_sql_string(source_pair)}'
        AS source_pair,


        'house_locality'
        AS strategy


    FROM

        {s1_table} a


    INNER JOIN

        {sx_table} b


        ON


        (


            a.house_locality_key
            <>
            ''


            AND


            a.house_locality_key
            =
            b.house_locality_key


        )


        OR


        (


            a.house_city_key
            <>
            ''


            AND


            a.house_city_key
            =
            b.house_city_key


        )


    WHERE


        (


            a.house_locality_key
            <>
            ''


            AND


            a.house_locality_key


            IN


            (


                SELECT

                    house_locality_key


                FROM

                    {s1_table}


                WHERE

                    house_locality_key
                    <>
                    ''


                GROUP BY

                    house_locality_key


                HAVING

                    COUNT(*)
                    <=
                    {MAX_BLOCK_SIZE}


            )


            AND


            b.house_locality_key


            IN


            (


                SELECT

                    house_locality_key


                FROM

                    {sx_table}


                WHERE

                    house_locality_key
                    <>
                    ''


                GROUP BY

                    house_locality_key


                HAVING

                    COUNT(*)
                    <=
                    {MAX_BLOCK_SIZE}


            )


        )


        OR


        (


            a.house_city_key
            <>
            ''


            AND


            a.house_city_key


            IN


            (


                SELECT

                    house_city_key


                FROM

                    {s1_table}


                WHERE

                    house_city_key
                    <>
                    ''


                GROUP BY

                    house_city_key


                HAVING

                    COUNT(*)
                    <=
                    {MAX_BLOCK_SIZE}


            )


            AND


            b.house_city_key


            IN


            (


                SELECT

                    house_city_key


                FROM

                    {sx_table}


                WHERE

                    house_city_key
                    <>
                    ''


                GROUP BY

                    house_city_key


                HAVING

                    COUNT(*)
                    <=
                    {MAX_BLOCK_SIZE}


            )


        );

    """


    execute_timed(

        con,

        sql,

        f"house_locality "
        f"[{source_pair}]"

    )


# ============================================================
# 32. TRAIN PAIRS
# ============================================================

train_pairs = [

    (
        "f_train_s2",
        "s1_s2"
    ),

    (
        "f_train_s3",
        "s1_s3"
    )

]


# ============================================================
# 33. TEST PAIRS
# ============================================================

test_pairs = [

    (
        "f_test_s2",
        "s1_s2"
    ),

    (
        "f_test_s3",
        "s1_s3"
    )

]


# ============================================================
# 34. TRAIN DETERMINISTIC BLOCKING
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "TRAIN DETERMINISTIC BLOCKING"

)

print(

    "============================================================"

)


# ============================================================
# STRATEGY 1 — EXACT NAME
# ============================================================

for sx_table, pair in train_pairs:

    add_exact_block(

        con,

        "f_train_s1",

        sx_table,

        "train_candidate_long",

        pair,

        "name_key_base",

        "exact_name"

    )


# ============================================================
# STRATEGY 2 — NAME + COUNTRY
# ============================================================

for sx_table, pair in train_pairs:

    add_exact_block(

        con,

        "f_train_s1",

        sx_table,

        "train_candidate_long",

        pair,

        "name_country_key",

        "name_country"

    )


# ============================================================
# STRATEGY 3 — NAME + POSTAL
# ============================================================

for sx_table, pair in train_pairs:

    add_exact_block(

        con,

        "f_train_s1",

        sx_table,

        "train_candidate_long",

        pair,

        "name_postal_key",

        "name_postal"

    )


# ============================================================
# STRATEGY 4 — HOUSE + LOCALITY/CITY
# ============================================================

for sx_table, pair in train_pairs:

    add_house_location_block(

        con,

        "f_train_s1",

        sx_table,

        "train_candidate_long",

        pair

    )


# ============================================================
# STRATEGY 5 — PHONETIC
# ============================================================

for sx_table, pair in train_pairs:

    add_exact_block(

        con,

        "f_train_s1",

        sx_table,

        "train_candidate_long",

        pair,

        "phonetic_key_base",

        "phonetic_name"

    )


# ============================================================
# STRATEGY 6 — ROMAN EXACT NAME
# ============================================================

for sx_table, pair in train_pairs:

    add_exact_block(

        con,

        "f_train_s1",

        sx_table,

        "train_candidate_long",

        pair,

        "roman_name_key_base",

        "roman_exact_name"

    )


# ============================================================
# 35. TEST DETERMINISTIC BLOCKING
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "TEST DETERMINISTIC BLOCKING"

)

print(

    "============================================================"

)


# ============================================================
# STRATEGY 1
# ============================================================

for sx_table, pair in test_pairs:

    add_exact_block(

        con,

        "f_test_s1",

        sx_table,

        "test_candidate_long",

        pair,

        "name_key_base",

        "exact_name"

    )


# ============================================================
# STRATEGY 2
# ============================================================

for sx_table, pair in test_pairs:

    add_exact_block(

        con,

        "f_test_s1",

        sx_table,

        "test_candidate_long",

        pair,

        "name_country_key",

        "name_country"

    )


# ============================================================
# STRATEGY 3
# ============================================================

for sx_table, pair in test_pairs:

    add_exact_block(

        con,

        "f_test_s1",

        sx_table,

        "test_candidate_long",

        pair,

        "name_postal_key",

        "name_postal"

    )


# ============================================================
# STRATEGY 4
# ============================================================

for sx_table, pair in test_pairs:

    add_house_location_block(

        con,

        "f_test_s1",

        sx_table,

        "test_candidate_long",

        pair

    )


# ============================================================
# STRATEGY 5
# ============================================================

for sx_table, pair in test_pairs:

    add_exact_block(

        con,

        "f_test_s1",

        sx_table,

        "test_candidate_long",

        pair,

        "phonetic_key_base",

        "phonetic_name"

    )


# ============================================================
# STRATEGY 6
# ============================================================

for sx_table, pair in test_pairs:

    add_exact_block(

        con,

        "f_test_s1",

        sx_table,

        "test_candidate_long",

        pair,

        "roman_name_key_base",

        "roman_exact_name"

    )


# ============================================================
# 36. SHOW DETERMINISTIC COUNTS
# ============================================================

print(

    "\n"
    "========== "
    "DETERMINISTIC COUNTS "
    "=========="

)


for table_name in [

    "train_candidate_long",

    "test_candidate_long"

]:

    count = con.execute(

        f"""

        SELECT COUNT(*)

        FROM
            {table_name}

        """

    ).fetchone()[0]


    print(

        f"{table_name}: "
        f"{count:,} rows"

    )


# ============================================================
# 37. GET UNRESOLVED S1
# ============================================================

def get_unresolved_s1(

    s1_feature_table,

    candidate_table

):

    sql = f"""

    SELECT

        a.entity_id,

        a.roman_name_key_base,

        a.tfidf_prefix2,

        a.country_key


    FROM

        {s1_feature_table} a


    LEFT JOIN


    (


        SELECT

            source1_entity_id,


            COUNT(
                DISTINCT
                candidate_entity_id
            )

            AS candidate_count


        FROM

            {candidate_table}


        GROUP BY

            source1_entity_id


    ) c


        ON


        a.entity_id
        =
        c.source1_entity_id


    WHERE


        COALESCE(

            c.candidate_count,

            0

        )


        <


        {MAX_DETERMINISTIC_CANDIDATES}


        AND


        a.roman_name_key_base
        <>
        '';

    """


    result = execute_timed(

        con,

        sql,

        f"Find unresolved S1 "
        f"in {candidate_table}"

    )


    return result.df()


# ============================================================
# 38. TF-IDF BLOCK
# ============================================================

def run_tfidf_block(

    s1_feature_table,

    sx_feature_table,

    candidate_table,

    source_pair

):

    print(

        "\n"
        "============================================================"

    )

    print(

        f"TF-IDF RETRIEVAL — "
        f"{source_pair}"

    )

    print(

        "============================================================"

    )


    unresolved = get_unresolved_s1(

        s1_feature_table,

        candidate_table

    )


    print(

        "\nS1 records requiring TF-IDF:",

        f"{len(unresolved):,}"

    )


    if unresolved.empty:

        print(

            "No unresolved S1 records."

        )

        return


    # --------------------------------------------------------
    # BUCKET
    # country + first 2 roman chars
    # --------------------------------------------------------

    unresolved[
        "country_key"
    ] = (

        unresolved[
            "country_key"
        ]

        .fillna("")

        .astype(str)

    )


    unresolved[
        "tfidf_prefix2"
    ] = (

        unresolved[
            "tfidf_prefix2"
        ]

        .fillna("")

        .astype(str)

    )


    unresolved[
        "bucket"
    ] = (

        unresolved[
            "country_key"
        ]

        +

        "¦"

        +

        unresolved[
            "tfidf_prefix2"
        ]

    )


    buckets = (

        unresolved[
            "bucket"
        ]

        .value_counts()

        .index

        .tolist()

    )


    print(

        "TF-IDF buckets:",

        f"{len(buckets):,}"

    )


    inserted = 0

    skipped_large = 0

    failed_buckets = 0

    start_time = time.time()


    # ========================================================
    # PROCESS EACH BUCKET
    # ========================================================

    for bucket_number, bucket in enumerate(

        buckets,

        start=1

    ):

        s1_bucket = unresolved[

            unresolved[
                "bucket"
            ]
            ==
            bucket

        ]


        if s1_bucket.empty:

            continue


        country, prefix = bucket.split(

            "¦",

            1

        )


        # ----------------------------------------------------
        # LOAD SOURCE BUCKET
        # ----------------------------------------------------

        source_df = con.execute(

            f"""

            SELECT

                entity_id,

                roman_name_key_base


            FROM

                {sx_feature_table}


            WHERE

                roman_name_key_base
                <>
                ''


                AND


                country_key = ?


                AND


                tfidf_prefix2 = ?

            """,

            [

                country,

                prefix

            ]

        ).df()


        if source_df.empty:

            continue


        # ----------------------------------------------------
        # LARGE BUCKET CHECK
        # ----------------------------------------------------

        if (

            len(source_df)

            >

            TFIDF_BUCKET_MAX_ROWS

        ):

            skipped_large += 1

            continue


        corpus = (

            source_df[
                "roman_name_key_base"
            ]

            .fillna("")

            .astype(str)

            .tolist()

        )


        queries = (

            s1_bucket[
                "roman_name_key_base"
            ]

            .fillna("")

            .astype(str)

            .tolist()

        )


        try:

            # ------------------------------------------------
            # TF-IDF
            # ------------------------------------------------

            vectorizer = TfidfVectorizer(

                analyzer="char",

                ngram_range=(

                    3,

                    5

                ),

                min_df=TFIDF_MIN_DF,

                max_features=
                TFIDF_MAX_FEATURES,

                lowercase=False,

                dtype=np.float32

            )


            X = vectorizer.fit_transform(

                corpus

            )


            Q = vectorizer.transform(

                queries

            )


            if X.shape[1] == 0:

                continue


            k = min(

                TFIDF_TOP_K,

                X.shape[0]

            )


            if k <= 0:

                continue


            # ------------------------------------------------
            # NEAREST NEIGHBORS
            # ------------------------------------------------

            nn = NearestNeighbors(

                metric="cosine",

                algorithm="brute",

                n_neighbors=k,

                n_jobs=-1

            )


            nn.fit(
                X
            )


            distances, indices = (

                nn.kneighbors(

                    Q,

                    return_distance=True

                )

            )


            candidate_rows = []


            bucket_s1_ids = (

                s1_bucket[
                    "entity_id"
                ]

                .astype(str)

                .tolist()

            )


            source_ids = (

                source_df[
                    "entity_id"
                ]

                .astype(str)

                .tolist()

            )


            # ------------------------------------------------
            # CONVERT NEIGHBORS TO ROWS
            # ------------------------------------------------

            for query_index in range(

                len(
                    bucket_s1_ids
                )

            ):

                source1_id = (

                    bucket_s1_ids[
                        query_index
                    ]

                )


                for neighbor_index in range(

                    indices.shape[1]

                ):

                    source_index = int(

                        indices[

                            query_index,

                            neighbor_index

                        ]

                    )


                    candidate_id = (

                        source_ids[
                            source_index
                        ]

                    )


                    similarity = (

                        1.0

                        -

                        float(

                            distances[

                                query_index,

                                neighbor_index

                            ]

                        )

                    )


                    if similarity <= 0:

                        continue


                    candidate_rows.append(

                        (

                            source1_id,

                            candidate_id,

                            source_pair,

                            "tfidf_char_topk"

                        )

                    )


            # ------------------------------------------------
            # INSERT
            # ------------------------------------------------

            if candidate_rows:

                temp_name = (

                    "tmp_tfidf_"

                    +

                    source_pair.replace(
                        "_",
                        ""
                    )

                    +

                    "_"

                    +

                    str(
                        bucket_number
                    )

                )


                temp_df = pd.DataFrame(

                    candidate_rows,

                    columns=[

                        "source1_entity_id",

                        "candidate_entity_id",

                        "source_pair",

                        "strategy"

                    ]

                )


                con.register(

                    temp_name,

                    temp_df

                )


                con.execute(

                    f"""

                    INSERT INTO
                        {candidate_table}

                    SELECT *

                    FROM
                        {temp_name}

                    """

                )


                con.unregister(

                    temp_name

                )


                inserted += len(

                    candidate_rows

                )


        except Exception as error:

            failed_buckets += 1


            print(

                "\nTF-IDF bucket error:"

            )


            print(

                "Bucket:",
                bucket

            )


            print(

                "Error:",
                error

            )


        del source_df

        gc.collect()


        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if (

            bucket_number % 50 == 0

            or

            bucket_number
            ==
            len(buckets)

        ):

            elapsed_minutes = (

                time.time()
                -
                start_time

            ) / 60


            print(

                f"Bucket "
                f"{bucket_number:,}/"
                f"{len(buckets):,}"
                f" | Inserted: "
                f"{inserted:,}"
                f" | Large skipped: "
                f"{skipped_large:,}"
                f" | Failed: "
                f"{failed_buckets:,}"
                f" | Time: "
                f"{elapsed_minutes:.1f} min"

            )


    print(

        "\nTF-IDF completed."

    )


    print(

        "Inserted candidates:",

        f"{inserted:,}"

    )


    print(

        "Skipped oversized buckets:",

        f"{skipped_large:,}"

    )


    print(

        "Failed buckets:",

        f"{failed_buckets:,}"

    )


# ============================================================
# 39. TRAIN TF-IDF S1 -> S2
# ============================================================

run_tfidf_block(

    "f_train_s1",

    "f_train_s2",

    "train_candidate_long",

    "s1_s2"

)


# ============================================================
# 40. TRAIN TF-IDF S1 -> S3
# ============================================================

run_tfidf_block(

    "f_train_s1",

    "f_train_s3",

    "train_candidate_long",

    "s1_s3"

)


# ============================================================
# 41. TEST TF-IDF S1 -> S2
# ============================================================

run_tfidf_block(

    "f_test_s1",

    "f_test_s2",

    "test_candidate_long",

    "s1_s2"

)


# ============================================================
# 42. TEST TF-IDF S1 -> S3
# ============================================================

run_tfidf_block(

    "f_test_s1",

    "f_test_s3",

    "test_candidate_long",

    "s1_s3"

)


# ============================================================
# 43. FINAL DEDUPLICATION
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "FINAL CANDIDATE DEDUPLICATION"

)

print(

    "============================================================"

)


for table in [

    "train_candidate_long",

    "test_candidate_long"

]:

    final_table = (

        table
        +
        "_final"

    )


    execute_timed(

        con,

        f"""

        CREATE OR REPLACE TABLE
            {final_table}

        AS

        SELECT DISTINCT

            source1_entity_id,

            candidate_entity_id,

            source_pair,

            strategy

        FROM
            {table};

        """,

        f"Deduplicate {table}"

    )


    count = con.execute(

        f"""

        SELECT COUNT(*)

        FROM
            {final_table}

        """

    ).fetchone()[0]


    print(

        f"{final_table}: "
        f"{count:,} rows"

    )


# ============================================================
# 44. STRATEGY AUDIT
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "STRATEGY AUDIT"

)

print(

    "============================================================"

)


for dataset_name in [

    "train",

    "test"

]:

    base_table = (

        f"{dataset_name}"
        f"_candidate_long_final"

    )


    audit_table = (

        f"{dataset_name}"
        f"_candidate_strategy_audit"

    )


    execute_timed(

        con,

        f"""

        CREATE OR REPLACE TABLE
            {audit_table}

        AS

        SELECT

            source_pair,

            strategy,

            COUNT(*)
            AS candidate_rows,

            COUNT(
                DISTINCT
                source1_entity_id
            )
            AS source1_with_candidates,

            COUNT(
                DISTINCT
                candidate_entity_id
            )
            AS unique_candidates

        FROM
            {base_table}

        GROUP BY

            source_pair,

            strategy

        ORDER BY

            source_pair,

            strategy;

        """,

        f"Strategy audit — "
        f"{dataset_name}"

    )


    audit_path = os.path.join(

        OUTPUT_DIR,

        f"{dataset_name}_candidate_strategy_audit.tsv"

    )


    con.execute(

        f"""

        COPY

            {audit_table}

        TO

            '{escape_sql_string(audit_path)}'

        (

            HEADER,

            DELIMITER '\\\\t'

        );

        """

    )


    print(

        "Saved:",

        audit_path

    )


# ============================================================
# 45. CANDIDATE COUNT DISTRIBUTIONS
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "CANDIDATE COUNT DISTRIBUTIONS"

)

print(

    "============================================================"

)


distribution_paths = {}


for dataset_name in [

    "train",

    "test"

]:

    candidate_table = (

        f"{dataset_name}"
        f"_candidate_long_final"

    )


    s1_table = (

        f"f_{dataset_name}_s1"

    )


    print(

        f"\nCalculating "
        f"{dataset_name} "
        f"candidate counts..."

    )


    distribution = con.execute(

        f"""

        SELECT

            a.entity_id
            AS source1_entity_id,


            COALESCE(

                COUNT(

                    DISTINCT
                    c.candidate_entity_id

                ),

                0

            )
            AS candidate_count


        FROM
            {s1_table} a


        LEFT JOIN
            {candidate_table} c


            ON


            a.entity_id
            =
            c.source1_entity_id


        GROUP BY

            a.entity_id


        ORDER BY

            a.entity_id;

        """

    ).df()


    print(

        f"\n{dataset_name.upper()} statistics:"

    )


    if not distribution.empty:

        print(

            distribution[
                "candidate_count"
            ]

            .describe(

                percentiles=[

                    0.50,

                    0.90,

                    0.95,

                    0.99

                ]

            )

            .to_string()

        )


    distribution_path = os.path.join(

        OUTPUT_DIR,

        f"{dataset_name}_candidate_count_distribution.tsv"

    )


    distribution.to_csv(

        distribution_path,

        sep="\t",

        index=False

    )


    distribution_paths[
        dataset_name
    ] = distribution_path


    print(

        "Saved:",

        distribution_path

    )


    del distribution

    gc.collect()


# ============================================================
# 46. TRAIN GROUND TRUTH PAIRS
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "TRAINING BLOCKING RECALL"

)

print(

    "============================================================"

)


execute_timed(

    con,

    """

    CREATE OR REPLACE TABLE
        train_gt_pairs

    AS

    SELECT

        CAST(

            source1_entity_id
            AS VARCHAR

        )

        AS source1_entity_id,


        TRIM(x)

        AS matched_entity_id


    FROM

        train_gt,


        UNNEST(

            string_split(

                COALESCE(

                    matched_entity_ids,

                    ''

                ),

                ','

            )

        )

        AS t(x)


    WHERE

        TRIM(x)
        <>
        '';

    """,

    "Expand ground truth pairs"

)


# ============================================================
# 47. TYPE GROUND TRUTH
# ============================================================

execute_timed(

    con,

    """

    CREATE OR REPLACE TABLE
        train_gt_pairs_typed

    AS

    SELECT

        g.source1_entity_id,


        g.matched_entity_id
        AS candidate_entity_id,


        CASE

            WHEN
                s2.entity_id IS NOT NULL

            THEN
                's1_s2'


            WHEN
                s3.entity_id IS NOT NULL

            THEN
                's1_s3'


            ELSE
                'unknown'

        END

        AS source_pair


    FROM
        train_gt_pairs g


    LEFT JOIN
        f_train_s2 s2


        ON


        g.matched_entity_id
        =
        s2.entity_id


    LEFT JOIN
        f_train_s3 s3


        ON


        g.matched_entity_id
        =
        s3.entity_id;

    """,

    "Type ground truth pairs"

)


unknown_gt = con.execute(

    """

    SELECT COUNT(*)

    FROM
        train_gt_pairs_typed

    WHERE
        source_pair
        =
        'unknown';

    """

).fetchone()[0]


print(

    "\nGround-truth IDs not found "
    "in S2/S3:",

    f"{unknown_gt:,}"

)


# ============================================================
# 48. PER-STRATEGY RECALL
# ============================================================

strategy_list = con.execute(

    """

    SELECT DISTINCT

        strategy

    FROM

        train_candidate_long_final

    ORDER BY

        strategy;

    """

).fetchall()


recall_records = []


for strategy_row in strategy_list:

    strategy = strategy_row[0]


    rows = con.execute(

        """

        SELECT

            g.source_pair,


            COUNT(*)
            AS gt_pairs,


            COUNT(
                c.candidate_entity_id
            )
            AS recovered_pairs,


            CASE

                WHEN COUNT(*) = 0

                THEN
                    0.0

                ELSE

                    COUNT(
                        c.candidate_entity_id
                    )::DOUBLE

                    /

                    COUNT(*)

            END

            AS recall


        FROM
            train_gt_pairs_typed g


        LEFT JOIN
            train_candidate_long_final c


            ON

            c.source1_entity_id
            =
            g.source1_entity_id


            AND


            c.candidate_entity_id
            =
            g.candidate_entity_id


            AND


            c.source_pair
            =
            g.source_pair


            AND


            c.strategy
            = ?


        WHERE

            g.source_pair
            <>
            'unknown'


        GROUP BY

            g.source_pair


        ORDER BY

            g.source_pair;

        """,

        [
            strategy
        ]

    ).fetchall()


    for row in rows:

        recall_records.append(

            {

                "strategy":
                    strategy,

                "source_pair":
                    row[0],

                "gt_pairs":
                    row[1],

                "recovered_pairs":
                    row[2],

                "recall":
                    row[3]

            }

        )


# ============================================================
# 49. UNION RECALL
# ============================================================

union_rows = con.execute(

    """

    SELECT

        g.source_pair,


        COUNT(*)
        AS gt_pairs,


        COUNT(
            c.candidate_entity_id
        )
        AS recovered_pairs,


        CASE

            WHEN COUNT(*) = 0

            THEN
                0.0

            ELSE

                COUNT(
                    c.candidate_entity_id
                )::DOUBLE

                /

                COUNT(*)

        END

        AS recall


    FROM
        train_gt_pairs_typed g


    LEFT JOIN


    (


        SELECT DISTINCT

            source1_entity_id,

            candidate_entity_id,

            source_pair

        FROM
            train_candidate_long_final


    ) c


        ON


        c.source1_entity_id
        =
        g.source1_entity_id


        AND


        c.candidate_entity_id
        =
        g.candidate_entity_id


        AND


        c.source_pair
        =
        g.source_pair


    WHERE

        g.source_pair
        <>
        'unknown'


    GROUP BY

        g.source_pair


    ORDER BY

        g.source_pair;

    """

).fetchall()


for row in union_rows:

    recall_records.append(

        {

            "strategy":
                "ALL_UNION",

            "source_pair":
                row[0],

            "gt_pairs":
                row[1],

            "recovered_pairs":
                row[2],

            "recall":
                row[3]

        }

    )


recall_df = pd.DataFrame(
    recall_records
)


# ============================================================
# 50. SAVE RECALL REPORT
# ============================================================

recall_path = os.path.join(

    OUTPUT_DIR,

    "train_blocking_recall_report.tsv"

)


recall_df.to_csv(

    recall_path,

    sep="\t",

    index=False

)


print(

    "\n"
    +
    recall_df.to_string(
        index=False
    )

)


print(

    "\nSaved:",

    recall_path

)


# ============================================================
# 51. CREATE TRAIN CANDIDATE FILE
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "CREATE TRAIN CANDIDATE FILE"

)

print(

    "============================================================"

)


train_candidate_path = os.path.join(

    OUTPUT_DIR,

    "train_candidate_pairs.tsv"

)


execute_timed(

    con,

    f"""

    COPY

    (

        SELECT

            a.entity_id
            AS source1_entity_id,


            COALESCE(

                string_agg(

                    DISTINCT

                    c.candidate_entity_id,

                    ','

                    ORDER BY

                    c.candidate_entity_id

                ),

                ''

            )

            AS candidate_entity_ids


        FROM

            f_train_s1 a


        LEFT JOIN

            train_candidate_long_final c


            ON


            a.entity_id
            =
            c.source1_entity_id


        GROUP BY

            a.entity_id


        ORDER BY

            a.entity_id

    )


    TO


        '{escape_sql_string(train_candidate_path)}'


    (

        HEADER,

        DELIMITER '\\\\t'

    );

    """,

    "Write train_candidate_pairs.tsv"

)


print(

    "Saved:",

    train_candidate_path

)


# ============================================================
# 52. CREATE TEST CANDIDATE FILE
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "CREATE TEST CANDIDATE FILE"

)

print(

    "============================================================"

)


test_candidate_path = os.path.join(

    OUTPUT_DIR,

    "test_candidate_pairs.tsv"

)


execute_timed(

    con,

    f"""

    COPY

    (

        SELECT

            a.entity_id
            AS source1_entity_id,


            COALESCE(

                string_agg(

                    DISTINCT

                    c.candidate_entity_id,

                    ','

                    ORDER BY

                    c.candidate_entity_id

                ),

                ''

            )

            AS candidate_entity_ids


        FROM

            f_test_s1 a


        LEFT JOIN

            test_candidate_long_final c


            ON


            a.entity_id
            =
            c.source1_entity_id


        GROUP BY

            a.entity_id


        ORDER BY

            a.entity_id

    )


    TO


        '{escape_sql_string(test_candidate_path)}'


    (

        HEADER,

        DELIMITER '\\\\t'

    );

    """,

    "Write test_candidate_pairs.tsv"

)


print(

    "Saved:",

    test_candidate_path

)


# ============================================================
# 53. COPY FINAL candidate_pairs.tsv
# ============================================================

candidate_submission_path = os.path.join(

    OUTPUT_DIR,

    "candidate_pairs.tsv"

)


print(

    "\nCopying test candidate file to:"

)


print(

    candidate_submission_path

)


with open(

    test_candidate_path,

    "rb"

) as src:

    with open(

        candidate_submission_path,

        "wb"

    ) as dst:

        while True:

            chunk = src.read(

                8 * 1024 * 1024

            )


            if not chunk:

                break


            dst.write(
                chunk
            )


print(

    "Final candidate_pairs.tsv created."

)


# ============================================================
# 54. FINAL SANITY CHECK
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "FINAL SANITY CHECK"

)

print(

    "============================================================"

)


test_s1_rows = con.execute(

    """

    SELECT COUNT(*)

    FROM
        f_test_s1

    """

).fetchone()[0]


candidate_file_rows = con.execute(

    f"""

    SELECT COUNT(*)

    FROM read_csv(

        '{escape_sql_string(candidate_submission_path)}',

        auto_detect=true,

        delim='\\\\t',

        header=true,

        quote='',

        all_varchar=true

    )

    """

).fetchone()[0]


duplicate_s1 = con.execute(

    f"""

    SELECT COUNT(*)

    FROM

    (

        SELECT

            source1_entity_id

        FROM read_csv(

            '{escape_sql_string(candidate_submission_path)}',

            auto_detect=true,

            delim='\\\\t',

            header=true,

            quote='',

            all_varchar=true

        )

        GROUP BY

            source1_entity_id

        HAVING

            COUNT(*) > 1

    )

    """

).fetchone()[0]


print(

    "Test S1 rows:",

    f"{test_s1_rows:,}"

)


print(

    "candidate_pairs.tsv rows:",

    f"{candidate_file_rows:,}"

)


print(

    "Duplicate S1 rows:",

    f"{duplicate_s1:,}"

)


if (

    test_s1_rows
    ==
    candidate_file_rows

):

    print(

        "Candidate row count: PASS"

    )

else:

    print(

        "Candidate row count does not "
        "equal current Test S1 row count."

    )


# ============================================================
# 55. TEST CANDIDATE COVERAGE
# ============================================================

test_s1_with_candidates = con.execute(

    """

    SELECT

        COUNT(
            DISTINCT source1_entity_id
        )

    FROM

        test_candidate_long_final

    """

).fetchone()[0]


test_total_s1 = con.execute(

    """

    SELECT COUNT(*)

    FROM
        f_test_s1

    """

).fetchone()[0]


test_without_candidates = (

    test_total_s1
    -
    test_s1_with_candidates

)


print(

    "\nTest S1 with at least one candidate:",

    f"{test_s1_with_candidates:,}"

)


print(

    "Test S1 without candidates:",

    f"{test_without_candidates:,}"

)


# ============================================================
# 56. SAVE SUMMARY
# ============================================================

summary = {

    "stage":
        "Stage 4 - Blocking / Candidate Generation",


    "output_dir":
        OUTPUT_DIR,


    "local_work_dir":
        LOCAL_WORK_DIR,


    "duckdb_database":
        DB_PATH,


    "duckdb_temp_directory":
        LOCAL_TEMP_DIR,


    "stage3_row_count_check":
        "DISABLED",


    "loader":
        "Explicit VARCHAR TSV schema; quote parsing disabled",


    "strategies":

        [

            "exact_name",

            "name_country",

            "name_postal",

            "house_locality",

            "phonetic_name",

            "roman_exact_name",

            "tfidf_char_topk"

        ],


    "max_block_size":
        MAX_BLOCK_SIZE,


    "tfidf_top_k":
        TFIDF_TOP_K,


    "tfidf_max_features":
        TFIDF_MAX_FEATURES,


    "tfidf_bucket_max_rows":
        TFIDF_BUCKET_MAX_ROWS,


    "tfidf_min_df":
        TFIDF_MIN_DF,


    "max_deterministic_candidates":
        MAX_DETERMINISTIC_CANDIDATES,


    "train_candidate_pairs":
        train_candidate_path,


    "test_candidate_pairs":
        test_candidate_path,


    "candidate_submission":
        candidate_submission_path,


    "train_blocking_recall_report":
        recall_path,


    "test_s1_rows":
        int(
            test_s1_rows
        ),


    "candidate_pairs_rows":
        int(
            candidate_file_rows
        ),


    "duplicate_s1_rows":
        int(
            duplicate_s1
        ),


    "test_s1_with_candidates":
        int(
            test_s1_with_candidates
        ),


    "test_s1_without_candidates":
        int(
            test_without_candidates
        )

}


summary_path = os.path.join(

    OUTPUT_DIR,

    "stage4_summary.json"

)


with open(

    summary_path,

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        summary,

        file,

        indent=2

    )


print(

    "\nSaved:",

    summary_path

)


# ============================================================
# 57. FINAL OUTPUTS
# ============================================================

print(

    "\n"
    "============================================================"

)

print(

    "STAGE 4 COMPLETE"

)

print(

    "============================================================"

)


print(

    "\nGoogle Drive output directory:"

)

print(
    OUTPUT_DIR
)


print(

    "\nGenerated files:"

)

print(
    "1. train_candidate_pairs.tsv"
)

print(
    "2. test_candidate_pairs.tsv"
)

print(
    "3. candidate_pairs.tsv"
)

print(
    "4. train_candidate_strategy_audit.tsv"
)

print(
    "5. test_candidate_strategy_audit.tsv"
)

print(
    "6. train_candidate_count_distribution.tsv"
)

print(
    "7. test_candidate_count_distribution.tsv"
)

print(
    "8. train_blocking_recall_report.tsv"
)

print(
    "9. stage4_summary.json"
)


print(

    "\nLocal DuckDB:"

)

print(
    DB_PATH
)


print(

    "\nDuckDB temporary directory:"

)

print(
    LOCAL_TEMP_DIR
)


print(

    "\nNext stage:"

)

print(

    "Stage 5 — Pair-level feature generation "
    "+ supervised entity matching"

)


print(

    "============================================================"

)


# ============================================================
# 58. CLOSE
# ============================================================

con.close()

gc.collect()


print(
    "\nDuckDB connection closed."
)

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 324, in run
    session = self.get_default_session(options)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/index_command.py", line 71, in get_default_session
    self._session = self.enter_context(self._build_session(options))
                                       ~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/index_command.py", line 100, in _build_session
    session = PipSession(
        cache=os.path.join(cache_dir, "http-v2") if cache_dir else None,
    ...<3 lines>...
        ssl_context=ssl_context,
    )
  Fil

TransactionException: TransactionContext Error: Catalog write-write conflict on alter with "Schema\0main\0main\0Table\0main\0train_s1"

In [10]:
# ============================================================
# AMAZON ML CHALLENGE 2026
# STAGE 4 — FINAL SANITY CHECK / FINALIZATION
# ============================================================
#
# Run this in a NEW Google Colab cell.
#
# The expensive Stage-4 blocking + TF-IDF has already run.
# This cell only validates and finalizes the existing outputs.
# ============================================================

import os
import json
import gc
import time
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/dataset"
)

S1_TEST = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of test_source1_stage3_normalized.tsv"
)

S2_TEST = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of test_source2_stage3_normalized.tsv"
)

S3_TEST = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of Copy of test_source3_stage3_normalized.tsv"
)

TEST_CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "test_candidate_pairs.tsv"
)

CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "candidate_pairs.tsv"
)

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "stage4_summary.json"
)

TRAIN_CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "train_candidate_pairs.tsv"
)

TRAIN_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "train_candidate_strategy_audit.tsv"
)

TEST_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "test_candidate_strategy_audit.tsv"
)

TRAIN_DIST_PATH = os.path.join(
    OUTPUT_DIR,
    "train_candidate_count_distribution.tsv"
)

TEST_DIST_PATH = os.path.join(
    OUTPUT_DIR,
    "test_candidate_count_distribution.tsv"
)

RECALL_PATH = os.path.join(
    OUTPUT_DIR,
    "train_blocking_recall_report.tsv"
)


# ============================================================
# 2. START
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "STAGE 4 FINAL SANITY CHECK"
)

print(
    "============================================================"
)


# ============================================================
# 3. CHECK REQUIRED FILES
# ============================================================

required_files = [

    S1_TEST,
    S2_TEST,
    S3_TEST,

    TEST_CANDIDATE_PATH,
    CANDIDATE_PATH

]

missing_files = [

    path

    for path in required_files

    if not os.path.exists(path)

]

if missing_files:

    print(
        "\nMissing files:"
    )

    for path in missing_files:

        print(
            " -",
            path
        )

    raise FileNotFoundError(
        "Required Stage-4 files are missing."
    )


print(
    "\nAll required files found."
)


# ============================================================
# 4. READ TEST S1 IDs ONLY
# ============================================================

print(
    "\nLoading Test S1 IDs..."
)

start = time.time()

test_s1 = pd.read_csv(

    S1_TEST,

    sep="\t",

    usecols=["entity_id"],

    dtype=str,

    keep_default_na=False

)

test_s1_ids = set(

    test_s1[
        "entity_id"
    ]
    .astype(str)

)

test_s1_rows = len(test_s1)

print(
    "Test S1 rows:",
    f"{test_s1_rows:,}"
)

print(
    "Time:",
    f"{(time.time() - start):.1f} sec"
)

del test_s1
gc.collect()


# ============================================================
# 5. READ FINAL candidate_pairs.tsv
# ============================================================

print(
    "\nLoading candidate_pairs.tsv..."
)

start = time.time()

candidate = pd.read_csv(

    CANDIDATE_PATH,

    sep="\t",

    dtype=str,

    keep_default_na=False,

    low_memory=False

)

print(
    "Candidate rows:",
    f"{len(candidate):,}"
)

print(
    "Columns:",
    list(candidate.columns)
)

print(
    "Time:",
    f"{(time.time() - start) / 60:.1f} min"
)


# ============================================================
# 6. HEADER CHECK
# ============================================================

expected_columns = [

    "source1_entity_id",

    "candidate_entity_ids"

]

actual_columns = list(
    candidate.columns
)

if actual_columns == expected_columns:

    print(
        "\nHeader check: PASS"
    )

else:

    print(
        "\nHeader check: FAIL"
    )

    print(
        "Expected:",
        expected_columns
    )

    print(
        "Actual:",
        actual_columns
    )

    raise ValueError(
        "candidate_pairs.tsv has an unexpected structure."
    )


# ============================================================
# 7. ROW COUNT CHECK
# ============================================================

print(
    "\n========== ROW COUNT =========="
)

candidate_rows = len(candidate)

print(
    "Test S1 rows:",
    f"{test_s1_rows:,}"
)

print(
    "candidate_pairs.tsv rows:",
    f"{candidate_rows:,}"
)


row_count_pass = (

    test_s1_rows
    ==
    candidate_rows

)


print(
    "Row count:",
    "PASS"
    if row_count_pass
    else
    "FAIL"
)


# ============================================================
# 8. DUPLICATE S1 CHECK
# ============================================================

print(
    "\n========== DUPLICATE S1 CHECK =========="
)

duplicate_mask = (

    candidate[
        "source1_entity_id"
    ]

    .duplicated(
        keep=False
    )

)

duplicate_rows = int(
    duplicate_mask.sum()
)

duplicate_ids = int(

    candidate.loc[
        duplicate_mask,
        "source1_entity_id"
    ]
    .nunique()

)


print(
    "Duplicate S1 rows:",
    f"{duplicate_rows:,}"
)

print(
    "Duplicate S1 IDs:",
    f"{duplicate_ids:,}"
)


duplicate_s1_pass = (
    duplicate_rows == 0
)


print(
    "Duplicate S1 check:",
    "PASS"
    if duplicate_s1_pass
    else
    "FAIL"
)


# ============================================================
# 9. S1 COVERAGE
# ============================================================

print(
    "\n========== S1 COVERAGE =========="
)

candidate_s1_ids = set(

    candidate[
        "source1_entity_id"
    ]
    .astype(str)

)

missing_s1 = (

    test_s1_ids
    -
    candidate_s1_ids

)

extra_s1 = (

    candidate_s1_ids
    -
    test_s1_ids

)


print(
    "Test S1 unique IDs:",
    f"{len(test_s1_ids):,}"
)

print(
    "Candidate unique S1 IDs:",
    f"{len(candidate_s1_ids):,}"
)

print(
    "Missing Test S1 IDs:",
    f"{len(missing_s1):,}"
)

print(
    "Extra S1 IDs:",
    f"{len(extra_s1):,}"
)


coverage_pass = (

    len(missing_s1) == 0

    and

    len(extra_s1) == 0

)


print(
    "S1 coverage:",
    "PASS"
    if coverage_pass
    else
    "FAIL"
)


if missing_s1:

    print(
        "\nFirst missing S1 IDs:"
    )

    for entity_id in sorted(
        list(missing_s1)
    )[:20]:

        print(
            " -",
            entity_id
        )


if extra_s1:

    print(
        "\nFirst extra S1 IDs:"
    )

    for entity_id in sorted(
        list(extra_s1)
    )[:20]:

        print(
            " -",
            entity_id
        )


# ============================================================
# 10. LOAD TEST S2/S3 IDs
# ============================================================

print(
    "\n========== LOADING TEST S2/S3 IDS =========="
)

start = time.time()

test_s2 = pd.read_csv(

    S2_TEST,

    sep="\t",

    usecols=["entity_id"],

    dtype=str,

    keep_default_na=False

)

test_s3 = pd.read_csv(

    S3_TEST,

    sep="\t",

    usecols=["entity_id"],

    dtype=str,

    keep_default_na=False

)


s2_ids = set(

    test_s2[
        "entity_id"
    ]
    .astype(str)

)

s3_ids = set(

    test_s3[
        "entity_id"
    ]
    .astype(str)

)

valid_candidate_ids = (

    s2_ids
    |
    s3_ids

)


print(
    "Test S2 IDs:",
    f"{len(s2_ids):,}"
)

print(
    "Test S3 IDs:",
    f"{len(s3_ids):,}"
)

print(
    "Valid S2/S3 candidate IDs:",
    f"{len(valid_candidate_ids):,}"
)

print(
    "Time:",
    f"{(time.time() - start) / 60:.1f} min"
)

del test_s2
del test_s3
gc.collect()


# ============================================================
# 11. VALIDATE CANDIDATE ENTITY IDS
# ============================================================

print(
    "\n========== CANDIDATE ID CHECK =========="
)

invalid_ids = set()

rows_with_candidates = 0

rows_without_candidates = 0

rows_with_invalid_ids = 0

rows_with_duplicate_ids = 0


for candidate_string in candidate[
    "candidate_entity_ids"
].astype(str):

    candidate_string = (
        candidate_string.strip()
    )


    if candidate_string == "":

        rows_without_candidates += 1

        continue


    rows_with_candidates += 1


    ids = [

        x.strip()

        for x in candidate_string.split(",")

        if x.strip()

    ]


    # duplicate IDs inside a single row
    if len(ids) != len(set(ids)):

        rows_with_duplicate_ids += 1


    row_invalid = False


    for entity_id in ids:

        if entity_id not in valid_candidate_ids:

            invalid_ids.add(
                entity_id
            )

            row_invalid = True


    if row_invalid:

        rows_with_invalid_ids += 1


print(
    "S1 rows with candidates:",
    f"{rows_with_candidates:,}"
)

print(
    "S1 rows without candidates:",
    f"{rows_without_candidates:,}"
)

print(
    "Rows with invalid candidate IDs:",
    f"{rows_with_invalid_ids:,}"
)

print(
    "Unique invalid candidate IDs:",
    f"{len(invalid_ids):,}"
)

print(
    "Rows with duplicate candidate IDs:",
    f"{rows_with_duplicate_ids:,}"
)


candidate_id_pass = (

    len(invalid_ids) == 0

)


internal_duplicate_pass = (

    rows_with_duplicate_ids == 0

)


print(
    "Candidate ID check:",
    "PASS"
    if candidate_id_pass
    else
    "FAIL"
)

print(
    "Internal duplicate check:",
    "PASS"
    if internal_duplicate_pass
    else
    "FAIL"
)


if invalid_ids:

    print(
        "\nFirst invalid candidate IDs:"
    )

    for entity_id in sorted(
        list(invalid_ids)
    )[:20]:

        print(
            " -",
            entity_id
        )


# ============================================================
# 12. CHECK test_candidate_pairs.tsv
# ============================================================

print(
    "\n========== TEST CANDIDATE FILE CHECK =========="
)


test_candidate = pd.read_csv(

    TEST_CANDIDATE_PATH,

    sep="\t",

    dtype=str,

    keep_default_na=False,

    low_memory=False

)


test_candidate_rows = len(
    test_candidate
)


print(
    "test_candidate_pairs.tsv rows:",
    f"{test_candidate_rows:,}"
)

print(
    "Columns:",
    list(test_candidate.columns)
)


test_candidate_structure_pass = (

    list(test_candidate.columns)
    ==
    expected_columns

)


test_candidate_row_pass = (

    test_candidate_rows
    ==
    test_s1_rows

)


print(
    "Structure:",
    "PASS"
    if test_candidate_structure_pass
    else
    "FAIL"
)

print(
    "Row count:",
    "PASS"
    if test_candidate_row_pass
    else
    "FAIL"
)


# ============================================================
# 13. COMPARE FINAL FILE WITH TEST CANDIDATE FILE
# ============================================================
#
# Compare using the two actual columns.
# ============================================================

print(
    "\n========== FINAL FILE CONSISTENCY =========="
)


same_rows = (

    len(candidate)
    ==
    len(test_candidate)

)

same_columns = (

    list(candidate.columns)
    ==
    list(test_candidate.columns)

)


if same_rows and same_columns:

    # Compare a compact hash instead of relying on DataFrame
    # object identity.

    import hashlib

    def dataframe_hash(df):

        h = hashlib.sha256()

        for chunk in range(
            0,
            len(df),
            10000
        ):

            block = df.iloc[
                chunk:chunk + 10000
            ]

            data = (
                block
                .to_csv(
                    sep="\t",
                    index=False,
                    header=False
                )
                .encode(
                    "utf-8"
                )
            )

            h.update(data)

        return h.hexdigest()


    candidate_hash = dataframe_hash(
        candidate
    )

    test_candidate_hash = dataframe_hash(
        test_candidate
    )


    same_content = (

        candidate_hash
        ==
        test_candidate_hash

    )

else:

    same_content = False


print(
    "candidate_pairs.tsv == "
    "test_candidate_pairs.tsv:",
    "PASS"
    if same_content
    else
    "CHECK"
)


del test_candidate
gc.collect()


# ============================================================
# 14. PREVIEW
# ============================================================

print(
    "\n========== OUTPUT PREVIEW =========="
)

print(

    candidate.head(10)
    .to_string(
        index=False
    )

)


# ============================================================
# 15. STAGE-4 OUTPUT FILES
# ============================================================

print(
    "\n========== STAGE 4 OUTPUTS =========="
)


stage4_outputs = {

    "train_candidate_pairs.tsv":
        TRAIN_CANDIDATE_PATH,

    "test_candidate_pairs.tsv":
        TEST_CANDIDATE_PATH,

    "candidate_pairs.tsv":
        CANDIDATE_PATH,

    "train_candidate_strategy_audit.tsv":
        TRAIN_AUDIT_PATH,

    "test_candidate_strategy_audit.tsv":
        TEST_AUDIT_PATH,

    "train_candidate_count_distribution.tsv":
        TRAIN_DIST_PATH,

    "test_candidate_count_distribution.tsv":
        TEST_DIST_PATH,

    "train_blocking_recall_report.tsv":
        RECALL_PATH

}


for filename, path in (
    stage4_outputs.items()
):

    print(

        f"{filename}:",

        "FOUND"

        if os.path.exists(path)

        else
        "MISSING"

    )


# ============================================================
# 16. OVERALL SANITY CHECK
# ============================================================

overall_pass = (

    row_count_pass

    and

    duplicate_s1_pass

    and

    coverage_pass

    and

    candidate_id_pass

    and

    internal_duplicate_pass

    and

    test_candidate_structure_pass

    and

    test_candidate_row_pass

    and

    same_content

)


print(
    "\n"
    "============================================================"
)

print(
    "OVERALL STAGE 4 SANITY CHECK:"
)

print(
    "PASS"
    if overall_pass
    else
    "CHECK REQUIRED"
)

print(
    "============================================================"
)


# ============================================================
# 17. REBUILD SUMMARY JSON
# ============================================================

summary = {

    "stage":
        "Stage 4 - Blocking / Candidate Generation",

    "output_dir":
        OUTPUT_DIR,

    "stage3_row_count_check":
        "DISABLED",

    "test_s1_rows":
        int(test_s1_rows),

    "candidate_pairs_rows":
        int(candidate_rows),

    "duplicate_s1_rows":
        int(duplicate_rows),

    "duplicate_s1_ids":
        int(duplicate_ids),

    "missing_test_s1_ids":
        int(len(missing_s1)),

    "extra_candidate_s1_ids":
        int(len(extra_s1)),

    "test_s1_with_candidates":
        int(rows_with_candidates),

    "test_s1_without_candidates":
        int(rows_without_candidates),

    "invalid_candidate_rows":
        int(rows_with_invalid_ids),

    "invalid_candidate_ids":
        int(len(invalid_ids)),

    "duplicate_candidate_rows":
        int(rows_with_duplicate_ids),

    "candidate_file_matches_test_candidate_file":
        bool(same_content),

    "final_sanity_check":
        "PASS"
        if overall_pass
        else
        "CHECK REQUIRED",

    "candidate_submission":
        CANDIDATE_PATH,

    "test_candidate_pairs":
        TEST_CANDIDATE_PATH,

    "train_candidate_pairs":
        TRAIN_CANDIDATE_PATH,

    "train_blocking_recall_report":
        RECALL_PATH

}


with open(

    SUMMARY_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=2

    )


print(
    "\nUpdated:",
    SUMMARY_PATH
)


# ============================================================
# 18. CLEAN UP
# ============================================================

del candidate
gc.collect()


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n"
    "============================================================"
)

if overall_pass:

    print(
        "STAGE 4 IS FULLY FINALIZED"
    )

    print(
        "\nNext stage:"
    )

    print(
        "STAGE 5 — PAIR-LEVEL FEATURE GENERATION"
    )

else:

    print(
        "STAGE 4 SANITY CHECK NEEDS ATTENTION"
    )

print(
    "============================================================"
)


STAGE 4 FINAL SANITY CHECK

All required files found.

Loading Test S1 IDs...
Test S1 rows: 800,000
Time: 14.5 sec

Loading candidate_pairs.tsv...
Candidate rows: 800,000
Columns: ['source1_entity_id\\candidate_entity_ids']
Time: 1.1 min

Header check: FAIL
Expected: ['source1_entity_id', 'candidate_entity_ids']
Actual: ['source1_entity_id\\candidate_entity_ids']


ValueError: candidate_pairs.tsv has an unexpected structure.

In [11]:
# ============================================================
# AMAZON ML CHALLENGE 2026
# STAGE 4 — OUTPUT TSV REPAIR
# ============================================================
#
# PROBLEM:
# Stage 4 created some TSV files using a literal backslash
# separator instead of a real TAB.
#
# Example current header:
#
# source1_entity_id\candidate_entity_ids
#
# Correct:
#
# source1_entity_id<TAB>candidate_entity_ids
#
# This script:
# 1. Repairs the Stage-4 generated TSV files
# 2. Preserves all data
# 3. Does NOT rerun blocking
# 4. Does NOT rerun TF-IDF
# 5. Re-runs the final sanity check
# ============================================================

import os
import shutil
import gc
import time


# ============================================================
# 1. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/dataset"
)


# ============================================================
# 2. STAGE-4 FILES TO REPAIR
# ============================================================
#
# Only files generated by Stage 4 are touched.
#
# Stage-3 input files are NOT touched.
# ============================================================

stage4_files = [

    "train_candidate_pairs.tsv",

    "test_candidate_pairs.tsv",

    "candidate_pairs.tsv",

    "train_candidate_strategy_audit.tsv",

    "test_candidate_strategy_audit.tsv",

    "train_candidate_count_distribution.tsv",

    "test_candidate_count_distribution.tsv",

    "train_blocking_recall_report.tsv",

]


# ============================================================
# 3. REPAIR FUNCTION
# ============================================================

def repair_stage4_tsv(path):

    print(
        "\n"
        "------------------------------------------------------------"
    )

    print(
        "Checking:",
        os.path.basename(path)
    )

    if not os.path.exists(path):

        print(
            "File not found — skipping."
        )

        return


    # --------------------------------------------------------
    # Read first line only
    # --------------------------------------------------------

    with open(
        path,
        "rb"
    ) as f:

        first_line = f.readline()


    print(
        "Original header bytes:",
        repr(first_line[:200])
    )


    # --------------------------------------------------------
    # Already correct
    # --------------------------------------------------------

    if b"\t" in first_line:

        print(
            "Real TAB detected."
        )

        print(
            "No repair required."
        )

        return


    # --------------------------------------------------------
    # Detect literal backslash-t
    # Example:
    #
    # source1_entity_id\\tcandidate_entity_ids
    # --------------------------------------------------------

    if b"\\t" in first_line:

        separator = b"\\t"

        print(
            "Detected literal \\\\t separator."
        )


    # --------------------------------------------------------
    # Detect literal backslash
    # Example from your error:
    #
    # source1_entity_id\\candidate_entity_ids
    # --------------------------------------------------------

    elif b"\\" in first_line:

        separator = b"\\"

        print(
            "Detected literal backslash separator."
        )


    else:

        print(
            "Could not identify the separator."
        )

        raise ValueError(
            f"Unknown TSV separator in {path}"
        )


    # ========================================================
    # TEMP FILE
    # ========================================================

    temp_path = (

        path
        +
        ".repairing"

    )


    start = time.time()

    line_count = 0

    repaired_count = 0


    # ========================================================
    # STREAMING REPAIR
    # ========================================================
    #
    # We process line-by-line so the 800k-row candidate file
    # does not need to be loaded entirely into RAM.
    # ========================================================

    with open(

        path,

        "rb"

    ) as src:

        with open(

            temp_path,

            "wb"

        ) as dst:

            for line in src:

                line_count += 1


                # ------------------------------------------------
                # Only replace the first separator.
                #
                # The candidate_entity_ids field contains commas,
                # not our separator.
                # ------------------------------------------------

                if separator in line:

                    line = line.replace(

                        separator,

                        b"\t",

                        1

                    )

                    repaired_count += 1


                dst.write(
                    line
                )


    # ========================================================
    # REPLACE ORIGINAL
    # ========================================================

    os.replace(

        temp_path,

        path

    )


    elapsed = (

        time.time()
        -
        start

    )


    print(
        "Lines processed:",
        f"{line_count:,}"
    )


    print(
        "Lines repaired:",
        f"{repaired_count:,}"
    )


    print(
        "Time:",
        f"{elapsed:.1f} sec"
    )


    # --------------------------------------------------------
    # Verify new header
    # --------------------------------------------------------

    with open(

        path,

        "rb"

    ) as f:

        new_header = f.readline()


    print(
        "New header bytes:",
        repr(new_header[:200])
    )


    if b"\t" in new_header:

        print(
            "REPAIR: PASS"
        )

    else:

        print(
            "REPAIR: FAILED"
        )

        raise ValueError(
            f"File still does not contain a real TAB: {path}"
        )


# ============================================================
# 4. REPAIR ALL STAGE-4 TSV FILES
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "REPAIRING STAGE-4 TSV OUTPUTS"
)

print(
    "============================================================"
)


for filename in stage4_files:

    path = os.path.join(

        OUTPUT_DIR,

        filename

    )

    repair_stage4_tsv(
        path
    )


# ============================================================
# 5. FINAL CANDIDATE FILE CHECK
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "VERIFYING candidate_pairs.tsv"
)

print(
    "============================================================"
)


candidate_path = os.path.join(

    OUTPUT_DIR,

    "candidate_pairs.tsv"

)


with open(

    candidate_path,

    "rb"

) as f:

    header = f.readline()


print(
    "Header:",
    repr(header)
)


if b"\t" not in header:

    raise ValueError(
        "candidate_pairs.tsv still does not contain "
        "a real TAB separator."
    )


# ============================================================
# 6. READ CANDIDATE FILE CORRECTLY
# ============================================================

print(
    "\nLoading candidate_pairs.tsv with pandas..."
)

start = time.time()


import pandas as pd


candidate = pd.read_csv(

    candidate_path,

    sep="\t",

    dtype=str,

    keep_default_na=False,

    low_memory=False

)


print(
    "Rows:",
    f"{len(candidate):,}"
)


print(
    "Columns:",
    list(candidate.columns)
)


print(
    "Time:",
    f"{(time.time() - start) / 60:.1f} min"
)


# ============================================================
# 7. HEADER CHECK
# ============================================================

expected_columns = [

    "source1_entity_id",

    "candidate_entity_ids"

]


if list(candidate.columns) == expected_columns:

    print(
        "\nHeader check: PASS"
    )

else:

    print(
        "\nHeader check: FAIL"
    )

    print(
        "Expected:",
        expected_columns
    )

    print(
        "Actual:",
        list(candidate.columns)
    )

    raise ValueError(
        "Candidate file structure is still incorrect."
    )


# ============================================================
# 8. READ TEST S1
# ============================================================

print(
    "\nLoading Test S1 IDs..."
)


S1_TEST = (

    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of test_source1_stage3_normalized.tsv"

)


test_s1 = pd.read_csv(

    S1_TEST,

    sep="\t",

    usecols=["entity_id"],

    dtype=str,

    keep_default_na=False

)


test_s1_ids = set(

    test_s1[
        "entity_id"
    ]

)


test_s1_rows = len(
    test_s1
)


del test_s1

gc.collect()


# ============================================================
# 9. ROW COUNT
# ============================================================

print(
    "\n========== ROW COUNT =========="
)


candidate_rows = len(
    candidate
)


print(
    "Test S1 rows:",
    f"{test_s1_rows:,}"
)


print(
    "candidate_pairs.tsv rows:",
    f"{candidate_rows:,}"
)


row_count_pass = (

    test_s1_rows
    ==
    candidate_rows

)


print(
    "Row count:",
    "PASS"
    if row_count_pass
    else
    "FAIL"
)


# ============================================================
# 10. DUPLICATE S1
# ============================================================

print(
    "\n========== DUPLICATE S1 CHECK =========="
)


duplicate_mask = (

    candidate[
        "source1_entity_id"
    ]

    .duplicated(
        keep=False
    )

)


duplicate_rows = int(
    duplicate_mask.sum()
)


duplicate_ids = int(

    candidate.loc[
        duplicate_mask,
        "source1_entity_id"
    ]
    .nunique()

)


print(
    "Duplicate S1 rows:",
    f"{duplicate_rows:,}"
)


print(
    "Duplicate S1 IDs:",
    f"{duplicate_ids:,}"
)


duplicate_pass = (

    duplicate_rows == 0

)


print(
    "Duplicate S1 check:",
    "PASS"
    if duplicate_pass
    else
    "FAIL"
)


# ============================================================
# 11. S1 COVERAGE
# ============================================================

print(
    "\n========== S1 COVERAGE =========="
)


candidate_s1_ids = set(

    candidate[
        "source1_entity_id"
    ]
    .astype(str)

)


missing_s1 = (

    test_s1_ids
    -
    candidate_s1_ids

)


extra_s1 = (

    candidate_s1_ids
    -
    test_s1_ids

)


print(
    "Test S1 IDs:",
    f"{len(test_s1_ids):,}"
)


print(
    "Candidate S1 IDs:",
    f"{len(candidate_s1_ids):,}"
)


print(
    "Missing S1 IDs:",
    f"{len(missing_s1):,}"
)


print(
    "Extra S1 IDs:",
    f"{len(extra_s1):,}"
)


coverage_pass = (

    len(missing_s1) == 0

    and

    len(extra_s1) == 0

)


print(
    "S1 coverage:",
    "PASS"
    if coverage_pass
    else
    "FAIL"
)


# ============================================================
# 12. LOAD TEST S2/S3 IDS
# ============================================================

print(
    "\n========== VALID CANDIDATE IDS =========="
)


S2_TEST = (

    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of test_source2_stage3_normalized.tsv"

)


S3_TEST = (

    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of Copy of test_source3_stage3_normalized.tsv"

)


test_s2 = pd.read_csv(

    S2_TEST,

    sep="\t",

    usecols=["entity_id"],

    dtype=str,

    keep_default_na=False

)


test_s3 = pd.read_csv(

    S3_TEST,

    sep="\t",

    usecols=["entity_id"],

    dtype=str,

    keep_default_na=False

)


s2_ids = set(

    test_s2[
        "entity_id"
    ]

)


s3_ids = set(

    test_s3[
        "entity_id"
    ]

)


valid_candidate_ids = (

    s2_ids
    |
    s3_ids

)


print(
    "Test S2 IDs:",
    f"{len(s2_ids):,}"
)


print(
    "Test S3 IDs:",
    f"{len(s3_ids):,}"
)


del test_s2
del test_s3

gc.collect()


# ============================================================
# 13. VALIDATE CANDIDATE IDs
# ============================================================

print(
    "\n"
    "========== CANDIDATE ID VALIDATION =========="
)


invalid_candidate_ids = set()

rows_with_candidates = 0

rows_without_candidates = 0

rows_with_invalid_ids = 0

rows_with_duplicate_candidate_ids = 0


for candidate_string in candidate[
    "candidate_entity_ids"
].astype(str):

    candidate_string = (

        candidate_string.strip()

    )


    if candidate_string == "":

        rows_without_candidates += 1

        continue


    rows_with_candidates += 1


    ids = [

        x.strip()

        for x in candidate_string.split(",")

        if x.strip()

    ]


    if len(ids) != len(set(ids)):

        rows_with_duplicate_candidate_ids += 1


    row_invalid = False


    for entity_id in ids:

        if entity_id not in valid_candidate_ids:

            invalid_candidate_ids.add(
                entity_id
            )

            row_invalid = True


    if row_invalid:

        rows_with_invalid_ids += 1


print(
    "S1 rows with candidates:",
    f"{rows_with_candidates:,}"
)


print(
    "S1 rows without candidates:",
    f"{rows_without_candidates:,}"
)


print(
    "Rows with invalid candidate IDs:",
    f"{rows_with_invalid_ids:,}"
)


print(
    "Unique invalid candidate IDs:",
    f"{len(invalid_candidate_ids):,}"
)


print(
    "Rows with duplicate candidate IDs:",
    f"{rows_with_duplicate_candidate_ids:,}"
)


candidate_id_pass = (

    len(invalid_candidate_ids) == 0

)


duplicate_candidate_pass = (

    rows_with_duplicate_candidate_ids == 0

)


print(
    "Candidate ID validation:",
    "PASS"
    if candidate_id_pass
    else
    "FAIL"
)


print(
    "Internal candidate duplicates:",
    "PASS"
    if duplicate_candidate_pass
    else
    "FAIL"
)


# ============================================================
# 14. VERIFY test_candidate_pairs.tsv
# ============================================================

print(
    "\n========== TEST CANDIDATE FILE =========="
)


test_candidate_path = os.path.join(

    OUTPUT_DIR,

    "test_candidate_pairs.tsv"

)


test_candidate = pd.read_csv(

    test_candidate_path,

    sep="\t",

    dtype=str,

    keep_default_na=False,

    low_memory=False

)


print(
    "Rows:",
    f"{len(test_candidate):,}"
)


print(
    "Columns:",
    list(test_candidate.columns)
)


test_candidate_pass = (

    list(test_candidate.columns)
    ==
    expected_columns

    and

    len(test_candidate)
    ==
    test_s1_rows

)


print(
    "test_candidate_pairs.tsv:",
    "PASS"
    if test_candidate_pass
    else
    "FAIL"
)


# ============================================================
# 15. COMPARE TEST + FINAL FILE
# ============================================================

print(
    "\n========== FINAL FILE CONSISTENCY =========="
)


same_content = (

    candidate.equals(
        test_candidate
    )

)


print(
    "candidate_pairs.tsv == "
    "test_candidate_pairs.tsv:",

    "PASS"
    if same_content
    else
    "CHECK"
)


del test_candidate

gc.collect()


# ============================================================
# 16. OVERALL
# ============================================================

overall_pass = (

    row_count_pass

    and

    duplicate_pass

    and

    coverage_pass

    and

    candidate_id_pass

    and

    duplicate_candidate_pass

    and

    test_candidate_pass

    and

    same_content

)


print(
    "\n"
    "============================================================"
)

print(
    "STAGE 4 FINAL RESULT"
)

print(
    "============================================================"
)


if overall_pass:

    print(
        "✅ STAGE 4 COMPLETE"
    )

    print(
        "\nCandidate file is ready for the next stage:"
    )

    print(
        CANDIDATE_PATH
    )

    print(
        "\nNEXT:"
    )

    print(
        "STAGE 5 — PAIR-LEVEL FEATURE GENERATION"
    )

else:

    print(
        "⚠️ STAGE 4 NEEDS ATTENTION"
    )


print(
    "============================================================"
)


# ============================================================
# 17. SAVE/UPDATE SUMMARY
# ============================================================

SUMMARY_PATH = os.path.join(

    OUTPUT_DIR,

    "stage4_summary.json"

)


try:

    with open(

        SUMMARY_PATH,

        "r",

        encoding="utf-8"

    ) as f:

        summary = json.load(f)

except Exception:

    summary = {}


summary.update(

    {

        "candidate_pairs_rows":
            int(candidate_rows),

        "test_s1_rows":
            int(test_s1_rows),

        "duplicate_s1_rows":
            int(duplicate_rows),

        "duplicate_s1_ids":
            int(duplicate_ids),

        "missing_test_s1_ids":
            int(len(missing_s1)),

        "extra_candidate_s1_ids":
            int(len(extra_s1)),

        "test_s1_with_candidates":
            int(rows_with_candidates),

        "test_s1_without_candidates":
            int(rows_without_candidates),

        "invalid_candidate_rows":
            int(rows_with_invalid_ids),

        "invalid_candidate_ids":
            int(len(invalid_candidate_ids)),

        "duplicate_candidate_rows":
            int(rows_with_duplicate_candidate_ids),

        "candidate_file_matches_test_candidate_file":
            bool(same_content),

        "final_sanity_check":
            (
                "PASS"
                if overall_pass
                else
                "CHECK REQUIRED"
            )

    }

)


with open(

    SUMMARY_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=2

    )


print(
    "\nUpdated:",
    SUMMARY_PATH
)


# ============================================================
# 18. CLEANUP
# ============================================================

del candidate

gc.collect()


REPAIRING STAGE-4 TSV OUTPUTS

------------------------------------------------------------
Checking: train_candidate_pairs.tsv
Original header bytes: b'source1_entity_id\\candidate_entity_ids\n'
Detected literal backslash separator.
Lines processed: 2,206,822
Lines repaired: 2,206,822
Time: 48.2 sec
New header bytes: b'source1_entity_id\tcandidate_entity_ids\n'
REPAIR: PASS

------------------------------------------------------------
Checking: test_candidate_pairs.tsv
Original header bytes: b'source1_entity_id\\candidate_entity_ids\n'
Detected literal backslash separator.
Lines processed: 800,001
Lines repaired: 800,001
Time: 104.6 sec
New header bytes: b'source1_entity_id\tcandidate_entity_ids\n'
REPAIR: PASS

------------------------------------------------------------
Checking: candidate_pairs.tsv
Original header bytes: b'source1_entity_id\\candidate_entity_ids\n'
Detected literal backslash separator.
Lines processed: 800,001
Lines repaired: 800,001
Time: 59.4 sec
New header byte

ValueError: Unknown TSV separator in /content/drive/MyDrive/Colab Notebooks/dataset/train_blocking_recall_report.tsv

In [12]:
# ============================================================
# AMAZON ML CHALLENGE 2026
# STAGE 4 — REPORT REGENERATION / FINALIZATION
# ============================================================
#
# IMPORTANT:
# The expensive Stage-4 blocking + TF-IDF has ALREADY run.
#
# This script:
#   1. DOES NOT rerun blocking
#   2. DOES NOT rerun TF-IDF
#   3. Uses the existing local DuckDB database
#   4. Regenerates the broken strategy-audit files
#   5. Regenerates the broken recall report
#   6. Re-checks candidate_pairs.tsv
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import time

import duckdb
import pandas as pd


# ============================================================
# 2. PATHS
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/dataset"
)

LOCAL_DB = (
    "/content/amazon_ml_stage4_work/"
    "stage4_blocking.duckdb"
)

LOCAL_TEMP = (
    "/content/amazon_ml_stage4_work/"
    "duckdb_temp"
)


TRAIN_S1 = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "train_normalized/"
    "Copy of train_source1_stage3_normalized.tsv"
)


TEST_S1 = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/"
    "Copy of test_source1_stage3_normalized.tsv"
)


CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "candidate_pairs.tsv"
)


TEST_CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "test_candidate_pairs.tsv"
)


TRAIN_CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "train_candidate_pairs.tsv"
)


TRAIN_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "train_candidate_strategy_audit.tsv"
)


TEST_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "test_candidate_strategy_audit.tsv"
)


TRAIN_DIST_PATH = os.path.join(
    OUTPUT_DIR,
    "train_candidate_count_distribution.tsv"
)


TEST_DIST_PATH = os.path.join(
    OUTPUT_DIR,
    "test_candidate_count_distribution.tsv"
)


RECALL_PATH = os.path.join(
    OUTPUT_DIR,
    "train_blocking_recall_report.tsv"
)


SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "stage4_summary.json"
)


# ============================================================
# 3. START
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "STAGE 4 REPORT REGENERATION"
)

print(
    "============================================================"
)


# ============================================================
# 4. CHECK DUCKDB
# ============================================================

if not os.path.exists(LOCAL_DB):

    raise FileNotFoundError(

        "The Stage-4 local DuckDB database was not found:\n"
        +
        LOCAL_DB
        +
        "\n\n"
        "If the Colab runtime was restarted, the local database "
        "was deleted and Stage 4 may need to be rerun."

    )


print(
    "\nDuckDB found:"
)

print(
    LOCAL_DB
)


# ============================================================
# 5. CONNECT
# ============================================================

os.makedirs(
    LOCAL_TEMP,
    exist_ok=True
)


con = duckdb.connect(
    LOCAL_DB
)


con.execute(
    "PRAGMA threads=2"
)


con.execute(
    "SET preserve_insertion_order=false"
)


con.execute(

    f"""
    SET temp_directory=
    '{LOCAL_TEMP.replace("'", "''")}'
    """

)


con.execute(
    "PRAGMA enable_progress_bar=false"
)


print(
    "\nConnected to DuckDB."
)


# ============================================================
# 6. CHECK EXISTING TABLES
# ============================================================

print(
    "\n========== DUCKDB TABLES =========="
)


tables = con.execute(

    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema='main'
    ORDER BY table_name
    """

).df()


print(
    tables.to_string(
        index=False
    )
)


required_tables = [

    "train_candidate_long_final",

    "test_candidate_long_final",

    "train_gt_pairs_typed"

]


existing_tables = set(

    tables[
        "table_name"
    ]
    .astype(str)

)


missing_tables = [

    table

    for table in required_tables

    if table not in existing_tables

]


if missing_tables:

    raise RuntimeError(

        "Required Stage-4 DuckDB tables are missing:\n"
        +
        "\n".join(
            missing_tables
        )

    )


# ============================================================
# 7. HELPER
# ============================================================

def query_df(
    sql,
    label
):

    print(
        f"\n[START] {label}"
    )

    start = time.time()

    df = con.execute(
        sql
    ).df()

    elapsed = (
        time.time()
        -
        start
    )

    print(
        f"[DONE] {label} "
        f"({elapsed / 60:.1f} min)"
    )

    return df


# ============================================================
# 8. REBUILD TRAIN STRATEGY AUDIT
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "REBUILDING TRAIN STRATEGY AUDIT"
)

print(
    "============================================================"
)


train_audit = query_df(

    """
    SELECT

        source_pair,

        strategy,

        COUNT(*) AS candidate_rows,

        COUNT(
            DISTINCT source1_entity_id
        ) AS source1_with_candidates,

        COUNT(
            DISTINCT candidate_entity_id
        ) AS unique_candidates

    FROM
        train_candidate_long_final

    GROUP BY

        source_pair,

        strategy

    ORDER BY

        source_pair,

        strategy

    """,

    "Train strategy audit"

)


train_audit.to_csv(

    TRAIN_AUDIT_PATH,

    sep="\t",

    index=False

)


print(
    "\nSaved:"
)

print(
    TRAIN_AUDIT_PATH
)


print(
    "\nTrain strategy audit:"
)

print(

    train_audit.to_string(
        index=False
    )

)


# ============================================================
# 9. REBUILD TEST STRATEGY AUDIT
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "REBUILDING TEST STRATEGY AUDIT"
)

print(
    "============================================================"
)


test_audit = query_df(

    """
    SELECT

        source_pair,

        strategy,

        COUNT(*) AS candidate_rows,

        COUNT(
            DISTINCT source1_entity_id
        ) AS source1_with_candidates,

        COUNT(
            DISTINCT candidate_entity_id
        ) AS unique_candidates

    FROM
        test_candidate_long_final

    GROUP BY

        source_pair,

        strategy

    ORDER BY

        source_pair,

        strategy

    """,

    "Test strategy audit"

)


test_audit.to_csv(

    TEST_AUDIT_PATH,

    sep="\t",

    index=False

)


print(
    "\nSaved:"
)

print(
    TEST_AUDIT_PATH
)


print(
    "\nTest strategy audit:"
)

print(

    test_audit.to_string(
        index=False
    )

)


# ============================================================
# 10. REBUILD GROUND-TRUTH PAIR TABLE IF NECESSARY
# ============================================================

gt_count = con.execute(

    """
    SELECT COUNT(*)
    FROM train_gt_pairs_typed
    """

).fetchone()[0]


print(
    "\nGround-truth typed pairs:",
    f"{gt_count:,}"
)


# ============================================================
# 11. PER-STRATEGY RECALL
# ============================================================
#
# More efficient than the original version:
#
# We join GT pairs against candidate pairs once and group
# by strategy/source_pair.
#
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "CALCULATING PER-STRATEGY RECALL"
)

print(
    "============================================================"
)


recall_by_strategy = query_df(

    """

    WITH gt AS (

        SELECT DISTINCT

            source1_entity_id,

            candidate_entity_id,

            source_pair

        FROM
            train_gt_pairs_typed

        WHERE
            source_pair <> 'unknown'

    ),

    hit AS (

        SELECT DISTINCT

            g.source1_entity_id,

            g.candidate_entity_id,

            g.source_pair,

            c.strategy

        FROM
            gt g

        INNER JOIN
            train_candidate_long_final c

            ON

            c.source1_entity_id
            =
            g.source1_entity_id

            AND

            c.candidate_entity_id
            =
            g.candidate_entity_id

            AND

            c.source_pair
            =
            g.source_pair

    ),

    gt_counts AS (

        SELECT

            source_pair,

            COUNT(*) AS gt_pairs

        FROM
            gt

        GROUP BY
            source_pair

    )

    SELECT

        h.source_pair,

        h.strategy,

        gc.gt_pairs,

        COUNT(*) AS recovered_pairs,

        CASE

            WHEN gc.gt_pairs = 0

            THEN 0.0

            ELSE

                COUNT(*)::DOUBLE
                /
                gc.gt_pairs

        END AS recall

    FROM
        hit h

    INNER JOIN
        gt_counts gc

        ON

        h.source_pair
        =
        gc.source_pair

    GROUP BY

        h.source_pair,

        h.strategy,

        gc.gt_pairs

    ORDER BY

        h.source_pair,

        h.strategy

    """,

    "Per-strategy recall"

)


# ============================================================
# 12. UNION RECALL
# ============================================================

print(
    "\nCalculating union recall..."
)


union_recall = query_df(

    """

    WITH gt AS (

        SELECT DISTINCT

            source1_entity_id,

            candidate_entity_id,

            source_pair

        FROM
            train_gt_pairs_typed

        WHERE
            source_pair <> 'unknown'

    ),

    candidates AS (

        SELECT DISTINCT

            source1_entity_id,

            candidate_entity_id,

            source_pair

        FROM
            train_candidate_long_final

    ),

    gt_counts AS (

        SELECT

            source_pair,

            COUNT(*) AS gt_pairs

        FROM
            gt

        GROUP BY
            source_pair

    ),

    hits AS (

        SELECT

            g.source_pair,

            COUNT(*) AS recovered_pairs

        FROM
            gt g

        INNER JOIN
            candidates c

            ON

            c.source1_entity_id
            =
            g.source1_entity_id

            AND

            c.candidate_entity_id
            =
            g.candidate_entity_id

            AND

            c.source_pair
            =
            g.source_pair

        GROUP BY
            g.source_pair

    )

    SELECT

        gc.source_pair,

        gc.gt_pairs,

        COALESCE(
            h.recovered_pairs,
            0
        ) AS recovered_pairs,

        CASE

            WHEN gc.gt_pairs = 0

            THEN 0.0

            ELSE

                COALESCE(
                    h.recovered_pairs,
                    0
                )::DOUBLE
                /
                gc.gt_pairs

        END AS recall

    FROM
        gt_counts gc

    LEFT JOIN
        hits h

        ON

        gc.source_pair
        =
        h.source_pair

    ORDER BY

        gc.source_pair

    """,

    "Union recall"

)


# ============================================================
# 13. COMBINE RECALL REPORT
# ============================================================

recall_records = []


for _, row in recall_by_strategy.iterrows():

    recall_records.append(

        {

            "strategy":
                row["strategy"],

            "source_pair":
                row["source_pair"],

            "gt_pairs":
                int(
                    row["gt_pairs"]
                ),

            "recovered_pairs":
                int(
                    row["recovered_pairs"]
                ),

            "recall":
                float(
                    row["recall"]
                )

        }

    )


for _, row in union_recall.iterrows():

    recall_records.append(

        {

            "strategy":
                "ALL_UNION",

            "source_pair":
                row["source_pair"],

            "gt_pairs":
                int(
                    row["gt_pairs"]
                ),

            "recovered_pairs":
                int(
                    row["recovered_pairs"]
                ),

            "recall":
                float(
                    row["recall"]
                )

        }

    )


recall_df = pd.DataFrame(

    recall_records,

    columns=[

        "strategy",

        "source_pair",

        "gt_pairs",

        "recovered_pairs",

        "recall"

    ]

)


# ============================================================
# 14. SAVE RECALL REPORT
# ============================================================

recall_df.to_csv(

    RECALL_PATH,

    sep="\t",

    index=False

)


print(
    "\n========== BLOCKING RECALL REPORT =========="
)


if recall_df.empty:

    print(
        "WARNING: Recall report is empty."
    )

else:

    print(

        recall_df.to_string(
            index=False
        )

    )


print(
    "\nSaved:"
)

print(
    RECALL_PATH
)


# ============================================================
# 15. VERIFY CANDIDATE PAIRS FILE
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "VERIFYING candidate_pairs.tsv"
)

print(
    "============================================================"
)


candidate = pd.read_csv(

    CANDIDATE_PATH,

    sep="\t",

    dtype=str,

    keep_default_na=False,

    low_memory=False

)


print(
    "Columns:",
    list(candidate.columns)
)

print(
    "Rows:",
    f"{len(candidate):,}"
)


# ============================================================
# 16. TEST S1 COUNT
# ============================================================

test_s1_count = con.execute(

    """
    SELECT COUNT(*)
    FROM f_test_s1
    """

).fetchone()[0]


candidate_rows = len(
    candidate
)


print(
    "\nTest S1 rows:",
    f"{test_s1_count:,}"
)


print(
    "Candidate rows:",
    f"{candidate_rows:,}"
)


if (
    test_s1_count
    ==
    candidate_rows
):

    print(
        "Row count: PASS"
    )

else:

    print(
        "Row count: FAIL"
    )


# ============================================================
# 17. DUPLICATE S1
# ============================================================

duplicate_s1 = int(

    candidate[
        "source1_entity_id"
    ]
    .duplicated(
        keep=False
    )
    .sum()

)


print(
    "Duplicate S1 rows:",
    f"{duplicate_s1:,}"
)


if duplicate_s1 == 0:

    print(
        "Duplicate S1: PASS"
    )

else:

    print(
        "Duplicate S1: FAIL"
    )


# ============================================================
# 18. VERIFY candidate_pairs.tsv HEADER
# ============================================================

expected_columns = [

    "source1_entity_id",

    "candidate_entity_ids"

]


if list(candidate.columns) == expected_columns:

    print(
        "Candidate header: PASS"
    )

else:

    print(
        "Candidate header: FAIL"
    )


# ============================================================
# 19. VERIFY test_candidate_pairs.tsv
# ============================================================

test_candidate = pd.read_csv(

    TEST_CANDIDATE_PATH,

    sep="\t",

    dtype=str,

    keep_default_na=False,

    low_memory=False

)


print(
    "\ntest_candidate_pairs.tsv rows:",
    f"{len(test_candidate):,}"
)


test_candidate_pass = (

    list(test_candidate.columns)
    ==
    expected_columns

    and

    len(test_candidate)
    ==
    test_s1_count

)


print(
    "test_candidate_pairs.tsv:",
    "PASS"
    if test_candidate_pass
    else
    "FAIL"
)


# ============================================================
# 20. UPDATE SUMMARY JSON
# ============================================================

summary = {

    "stage":
        "Stage 4 - Blocking / Candidate Generation",

    "stage3_row_count_check":
        "DISABLED",

    "candidate_submission":
        CANDIDATE_PATH,

    "test_candidate_pairs":
        TEST_CANDIDATE_PATH,

    "train_candidate_pairs":
        TRAIN_CANDIDATE_PATH,

    "train_strategy_audit":
        TRAIN_AUDIT_PATH,

    "test_strategy_audit":
        TEST_AUDIT_PATH,

    "train_candidate_count_distribution":
        TRAIN_DIST_PATH,

    "test_candidate_count_distribution":
        TEST_DIST_PATH,

    "train_blocking_recall_report":
        RECALL_PATH,

    "test_s1_rows":
        int(
            test_s1_count
        ),

    "candidate_pairs_rows":
        int(
            candidate_rows
        ),

    "duplicate_s1_rows":
        int(
            duplicate_s1
        ),

    "recall_rows":
        int(
            len(recall_df)
        ),

    "final_sanity_check":
        "PASS"
        if (
            test_s1_count
            ==
            candidate_rows
            and
            duplicate_s1 == 0
            and
            list(candidate.columns)
            ==
            expected_columns
            and
            test_candidate_pass
            and
            len(recall_df) > 0
        )
        else
        "CHECK REQUIRED"

}


with open(

    SUMMARY_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=2

    )


print(
    "\nUpdated:"
)

print(
    SUMMARY_PATH
)


# ============================================================
# 21. FINAL RESULT
# ============================================================

final_pass = (

    test_s1_count
    ==
    candidate_rows

    and

    duplicate_s1
    ==
    0

    and

    list(candidate.columns)
    ==
    expected_columns

    and

    test_candidate_pass

    and

    len(recall_df)
    >
    0

)


print(
    "\n"
    "============================================================"
)

if final_pass:

    print(
        "✅ STAGE 4 FINALIZATION COMPLETE"
    )

    print(
        "\nStage 4 candidate generation is finished."
    )

    print(
        "\nNext stage:"
    )

    print(
        "STAGE 5 — PAIR-LEVEL FEATURE GENERATION"
    )

else:

    print(
        "⚠️ STAGE 4 FINALIZATION NEEDS ATTENTION"
    )

print(
    "============================================================"
)


# ============================================================
# 22. CLOSE
# ============================================================

con.close()

del candidate
del train_audit
del test_audit
del recall_by_strategy
del union_recall
del recall_df

gc.collect()

print(
    "\nDuckDB connection closed."
)


STAGE 4 REPORT REGENERATION


FileNotFoundError: The Stage-4 local DuckDB database was not found:
/content/amazon_ml_stage4_work/stage4_blocking.duckdb

If the Colab runtime was restarted, the local database was deleted and Stage 4 may need to be rerun.

In [13]:
# ============================================================
# STAGE 4 — FINAL FILE VERIFICATION
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/dataset"

TEST_S1 = (
    "/content/drive/MyDrive/Colab Notebooks/dataset/"
    "test_normalized/Copy of test_source1_stage3_normalized.tsv"
)

CANDIDATE = os.path.join(
    OUTPUT_DIR,
    "candidate_pairs.tsv"
)

TEST_CANDIDATE = os.path.join(
    OUTPUT_DIR,
    "test_candidate_pairs.tsv"
)

TRAIN_CANDIDATE = os.path.join(
    OUTPUT_DIR,
    "train_candidate_pairs.tsv"
)


print("============================================================")
print("STAGE 4 FINAL FILE VERIFICATION")
print("============================================================")


# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

for path in [
    CANDIDATE,
    TEST_CANDIDATE,
    TRAIN_CANDIDATE
]:

    print(
        os.path.basename(path),
        "->",
        "FOUND" if os.path.exists(path) else "MISSING"
    )


# ------------------------------------------------------------
# Read Test S1 count
# ------------------------------------------------------------

print("\nReading Test S1 row count...")

test_s1_rows = sum(
    1
    for _ in open(
        TEST_S1,
        "r",
        encoding="utf-8",
        errors="replace"
    )
) - 1

print(
    "Test S1 rows:",
    f"{test_s1_rows:,}"
)


# ------------------------------------------------------------
# Check candidate header
# ------------------------------------------------------------

with open(
    CANDIDATE,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    header = f.readline().rstrip("\r\n")


print("\nCandidate header:")
print(repr(header))


expected = "source1_entity_id\tcandidate_entity_ids"


if header == expected:

    print("Header: PASS")

else:

    print("Header: FAIL")

    print("Expected:")
    print(repr(expected))


# ------------------------------------------------------------
# Count candidate rows
# ------------------------------------------------------------

print("\nCounting candidate rows...")

candidate_rows = sum(
    1
    for _ in open(
        CANDIDATE,
        "r",
        encoding="utf-8",
        errors="replace"
    )
) - 1

print(
    "candidate_pairs.tsv rows:",
    f"{candidate_rows:,}"
)


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print(
    "\n============================================================"
)

if (
    candidate_rows
    ==
    test_s1_rows
    and
    header == expected
):

    print(
        "✅ STAGE 4 CANDIDATE FILE VERIFIED"
    )

    print(
        "\nProceed to:"
    )

    print(
        "STAGE 5 — PAIR-LEVEL FEATURE GENERATION"
    )

else:

    print(
        "⚠️ Candidate file needs checking"
    )

print(
    "============================================================"
)

STAGE 4 FINAL FILE VERIFICATION
candidate_pairs.tsv -> FOUND
test_candidate_pairs.tsv -> FOUND
train_candidate_pairs.tsv -> FOUND

Reading Test S1 row count...
Test S1 rows: 800,000

Candidate header:
'source1_entity_id\tcandidate_entity_ids'
Header: PASS

Counting candidate rows...
candidate_pairs.tsv rows: 800,000

✅ STAGE 4 CANDIDATE FILE VERIFIED

Proceed to:
STAGE 5 — PAIR-LEVEL FEATURE GENERATION
